# Agent智能体
参考文档：https://hello-agents.datawhale.cc/

## 思维导图

```mermaid
mindmap
  root((Agent 智能体学习笔记))
    一、基础概念
      智能体定义
      大模型 vs Agent
      应用示例
    二、基础实现
      环境依赖安装
      工具函数开发
      LLM 本地加载
      Agent 行动循环
    三、Agent 思维范式
      ReAct 范式
      Plan-and-Solve 规划执行
      Reflection 反思纠错
    四、主流搭建平台
      Coze 零代码平台
      Dify 开源企业平台
      n8n 工作流自动化
      通用 Agent 产品
    五、上下文与记忆系统
      记忆系统原理
      记忆四大能力
      向量库记忆实现
    六、RAG 检索增强
      传统 RAG 流程
      Agentic RAG 智能检索
    七、协议与技能
      MCP 模型上下文协议
      Tool 工具 & Skill 技能
    八、Harness 驾驭工程
      沙箱安全隔离
      任务路由调度
      可观测性
      质量评估
      Loop 循环工程
    九、主流开发框架
      单智能体框架
      多智能体框架

```mermaid
mindmap
  root((Agent 智能体))
    ## 基础介绍
      智能体定义
      普通大模型 vs Agent
      典型使用场景
    ## 最简 Agent 实现
      依赖库安装
        requests
        openai
        tavily-python
      系统指令模板
      工具开发
        天气查询工具
        景点推荐工具
      工具箱封装
      本地LLM加载(Qwen3)
      Agent 行动循环演示
    ## Agent 思维范式
      ReAct 范式
        Thought-Action-Observation
        输出解析
        工具解析
        ReActAgent 实现
      Plan-and-Solve
        规划阶段
        执行阶段
      Reflection 反思机制
        执行
        反思
        优化
    ## 智能体搭建平台
      Coze 字节扣子
      Dify 开源LLM平台
      n8n 工作流自动化
      通用AI智能体
        MiniMax Agent
        智谱Agent
        Trae SOLO
    ## 上下文&记忆系统
      上下文工程概念
      记忆系统痛点
      记忆工作流程
      轻量级记忆实现
        存储remember
        检索recall
        整合consolidate
        遗忘forget
      进阶记忆
        图记忆
        多模态记忆
    ## RAG 检索增强生成
      RAG 核心原理
      文档处理流程
        加载PDF
        文本切分
        Embedding向量化
        Chroma向量库
      本地模型接入LangChain
      传统RAG演示
      Agentic RAG 智能检索
    ## MCP 模型上下文协议
      C/S架构
      服务端工具定义
      客户端调用流程
    ## Tool & Skill
      Tool 原子工具
      Skill 组合工作流
      热搜整合Skill示例
    ## Harness 驾驭工程
      沙箱安全隔离
        OS级沙箱
        容器级沙箱
        进程级沙箱
        语言级沙箱
        firejail沙箱实践
      路由调度
        路由定义
        简单路由代码
      可观测性
        追踪Traces
        日志Logs
        指标Metrics
      结果评估体系
      Loop 循环工程
    ## 主流开发框架
      单智能体框架
        LangGraph
        AgentScope
      多智能体框架
        AutoGen
        CAMEL

## 基本介绍

在人工智能领域，智能体被定义为任何能够通过传感器（Sensors）感知其所处环境（Environment），并自主地通过执行器（Actuators）采取行动（Action）以达成特定目标的实体。

| 特性 |	普通大模型 |	Agent|
|---|---|---|
|交互方式 |	一问一答，被动 |	给定目标，自主推进 |
|信息来源 |	依赖训练时的旧知识|	可通过工具获取实时信息|
|行动能力 |	只能输出文字|	可以调用API执行真实动作|
|例子 |	“告诉我今天大概穿什么”|	“帮我查今天北京天气，并下单买把伞”|


✈️ 场景：旅行规划师

```mermaid
graph LR
    U4["👤 用户：</br>周末去苏州玩，帮我规划"] --> A4["🧠 思考：先查周末天气"]
    A4 --> B4["🔧 行动：</br>GetWeather[苏州, 周末]"]
    B4 --> C4["👀 观察：</br>周六小雨，周日晴"]
    C4 --> D4["🧠 思考：</br>雨天推荐室内景点，</br>晴天推荐户外"]
    D4 --> E4["🔧 行动：</br>Search[苏州雨天室内景点]"]
    E4 --> F4["👀 观察：</br>博物馆、拙政园..."]
    F4 --> G4["🧠 思考：</br>信息够了，整合行程"]
    G4 --> H4["✅ 输出：</br>周六博物馆 + </br>周日拙政园 + 住宿推荐"]
    
    classDef user fill:#e0f2fe,stroke:#0284c7,color:#000
    classDef think fill:#fef3c7,stroke:#d97706,color:#000
    classDef action fill:#dcfce7,stroke:#16a34a,color:#000
    classDef observe fill:#f3e8ff,stroke:#9333ea,color:#000
    classDef output fill:#fecaca,stroke:#dc2626,color:#000

    class U4 user
    class A4,D4,G4 think
    class B4,E4,H4 action
    class C4,F4 observe
    class G4,H4 output

## 实现最简单的智能体

```mermaid
flowchart LR
    subgraph Agent
        direction TB
        a2(LLM) <--> a4(Tool)
        a2(LLM) <--> a5(Memory)
    end
    a1(input) --> Agent
    Agent --> a3(output)

### 安装所需的库

In [1]:
#!pip install requests openai
!pip show requests openai

Name: requests
Version: 2.34.2
Summary: Python HTTP for Humans.
Home-page: 
Author: 
Author-email: Kenneth Reitz <me@kennethreitz.org>
License: Apache-2.0
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: certifi, charset_normalizer, idna, urllib3
Required-by: datasets, flashinfer-python, gguf, jupyterlab_server, mistral_common, modelscope, opentelemetry-exporter-otlp-proto-http, ray, tavily-python, tiktoken, vllm
---
Name: openai
Version: 2.37.0
Summary: The official Python library for the openai API
Home-page: https://github.com/openai/openai-python
Author: 
Author-email: OpenAI <support@openai.com>
License: Apache-2.0
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: anyio, distro, httpx, jiter, pydantic, sniffio, tqdm, typing-extensions
Required-by: vllm


tavily-python是一个强大的 AI 搜索 API 客户端，用于获取实时的网络搜索结果，可以在[官网](https://app.tavily.com/home)注册后获取 API

In [2]:
# !pip install tavily-python 
!pip show tavily-python

Name: tavily-python
Version: 0.7.25
Summary: Python wrapper for the Tavily API
Home-page: https://github.com/tavily-ai/tavily-python
Author: Tavily AI
Author-email: support@tavily.com
License: 
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: httpx, requests, tiktoken
Required-by: 


In [3]:
# 设置你的apikey
%env TAVILY_API_KEY=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

env: TAVILY_API_KEY=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


### 系统指令模版

In [36]:
AGENT_SYSTEM_PROMPT = """
你是一个智能旅行助手。你的任务是分析用户的请求，并使用可用工具一步步地解决问题。

# 可用工具:
- `get_weather(city: str)`: 查询指定城市的实时天气。
- `get_attraction(city: str, weather: str)`: 根据城市和天气搜索推荐的旅游景点。

# 输出格式要求:
你的每次回复必须严格遵循以下格式，包含一对Thought和Action：

Thought: [你的思考过程和下一步计划]
Action: [你要执行的具体行动]

Action的格式必须是以下之一！：
1. 调用工具：function_name(arg_name="arg_value")，例如：get_weather(city="北京") 。
2. 结束任务：Finish[你的最终答案内容]。

# 重要提示:
- 每次只输出一对Thought-Action
- Action必须在同一行，不要换行
- 当收集到足够信息可以回答用户问题时，必须使用 Action: Finish[你的最终答案内容] 格式结束

请开始吧！
"""

### 工具1：查询真实天气
使用免费天气api服务 wttr.in

In [5]:
import requests

def get_weather(city: str) -> str:
    """
    通过调用 wttr.in API 查询今天真实的天气信息。
    """
    print("正在调用tool:get_weathern......")
    # API端点，我们请求JSON格式的数据
    url = f"https://wttr.in/{city}?format=j1"
    
    try:
        # 发起网络请求
        response = requests.get(url)
        # 检查响应状态码是否为200 (成功)
        response.raise_for_status() 
        # 解析返回的JSON数据
        data = response.json()
            
        # 提取当前天气状况
        current_condition = data['current_condition'][0]
        weather_desc = current_condition['weatherDesc'][0]['value']
        temp_c = current_condition['temp_C']
        # print("今天天气内容：",current_condition)
        
        # 格式化成自然语言返回
        return f"{city}当前天气:{weather_desc}，气温{temp_c}摄氏度"
        
    except requests.exceptions.RequestException as e:
        # 处理网络错误
        return f"错误:查询天气时遇到网络问题 - {e}"
    except (KeyError, IndexError) as e:
        # 处理数据解析错误
        return f"错误:解析天气数据失败，可能是城市名称无效 - {e}"

In [6]:
get_weather("苏州")

正在调用tool:get_weathern......


'苏州当前天气:Partly Cloudy ，气温32摄氏度'

### 工具2：搜索并推荐旅游景点

In [7]:
import os
from tavily import TavilyClient

def get_attraction(city: str, weather: str) -> str:
    """
    根据城市和天气，使用Tavily Search API搜索并返回优化后的景点推荐。
    """
    # 1. 从环境变量中读取API密钥
    print("正在调用tool:get_attraction......")
    api_key = os.environ.get("TAVILY_API_KEY")
    if not api_key:
        return "错误:未配置TAVILY_API_KEY环境变量。"

    # 2. 初始化Tavily客户端
    tavily = TavilyClient(api_key=api_key)
    
    # 3. 构造一个精确的查询
    query = f"'{city}' 在'{weather}'天气下最值得去的旅游景点推荐及理由"
    
    try:
        # 4. 调用API，include_answer=True会返回一个综合性的回答
        response = tavily.search(query=query, search_depth="basic", include_answer=True)
        
        # 5. Tavily返回的结果已经非常干净，可以直接使用
        # response['answer'] 是一个基于所有搜索结果的总结性回答
        if response.get("answer"):
            return response["answer"]
        
        # 如果没有综合性回答，则格式化原始结果
        formatted_results = []
        for result in response.get("results", []):
            formatted_results.append(f"- {result['title']}: {result['content']}")
        
        if not formatted_results:
             return "抱歉，没有找到相关的旅游景点推荐。"

        return "根据搜索，为您找到以下信息:\n" + "\n".join(formatted_results)

    except Exception as e:
        return f"错误:执行Tavily搜索时出现问题 - {e}"

In [9]:
get_attraction("苏州", get_weather("苏州"))

正在调用tool:get_weathern......
正在调用tool:get_attraction......


"Under partly cloudy skies with a temperature of 32°C, visit the Humble Administrator's Garden for a serene experience. This historic garden offers a peaceful retreat and showcases classical Chinese garden design."

### 将工具都放在工具箱里

In [10]:
# 将所有工具函数放入一个字典，方便后续调用
available_tools = {
    "get_weather": get_weather,
    "get_attraction": get_attraction,
}

### 使用LLM推理

In [13]:
import torch
import re
import gc
from modelscope import AutoModelForCausalLM, AutoTokenizer

def CleanMemory():
    torch.cuda.empty_cache()
    gc.collect()

#### 使用Qwen/Qwen3-0.6B本地模型推理，在6_LMandLora.ipynb里已经下载过了

In [12]:
!modelscope download --model Qwen/Qwen3-0.6B --local_dir ./Qwen/Qwen3-0.6B


 _   .-')                _ .-') _     ('-.             .-')                              _ (`-.    ('-.
( '.( OO )_             ( (  OO) )  _(  OO)           ( OO ).                           ( (OO  ) _(  OO)
 ,--.   ,--.).-'),-----. \     .'_ (,------.,--.     (_)---\_)   .-----.  .-'),-----.  _.`     \(,------.
 |   `.'   |( OO'  .-.  ',`'--..._) |  .---'|  |.-') /    _ |   '  .--./ ( OO'  .-.  '(__...--'' |  .---'
 |         |/   |  | |  ||  |  \  ' |  |    |  | OO )\  :` `.   |  |('-. /   |  | |  | |  /  | | |  |
 |  |'.'|  |\_) |  |\|  ||  |   ' |(|  '--. |  |`-' | '..`''.) /_) |OO  )\_) |  |\|  | |  |_.' |(|  '--.
 |  |   |  |  \ |  | |  ||  |   / : |  .--'(|  '---.'.-._)   \ ||  |`-'|   \ |  | |  | |  .___.' |  .--'
 |  |   |  |   `'  '-'  '|  '--'  / |  `---.|      | \       /(_'  '--'\    `'  '-'  ' |  |      |  `---.
 `--'   `--'     `-----' `-------'  `------'`------'  `-----'    `-----'      `-----'  `--'      `------'


Successfully Downloaded from model Qwen/Qwen3-0.6B.


In [11]:
def load_model(model_name="./Qwen/Qwen3-0.6B"):
    # 加载分词器
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # 加载模型
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.bfloat16,
        # dtype=torch.float16, #上面的不能用就用下面的
        device_map="auto"
    )
    return model, tokenizer

In [14]:
model, tokenizer = load_model("./Qwen/Qwen3-0.6B")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

#### 输入提示词

In [43]:
user_prompt = "你好，请帮我查询一下今天苏州的天气，然后根据天气推荐一个合适的旅游景点。"
prompt_history = [f"用户请求: {user_prompt}"]

print(f"用户输入: {user_prompt}")

用户输入: 你好，请帮我查询一下今天苏州的天气，然后根据天气推荐一个合适的旅游景点。


In [12]:
def get_qwen_output(messages, model, tokenizer):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True # 是否使用思考模式
    )
    # 编码
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    # 推理
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=32768 #最大上下文长度
    )
    # 得到输出并转 ids
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    try:
        # 找到思考结束的符号的id的位置 151668 (</think>)
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0
    # 思考内容
    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    # 输出内容
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
    return thinking_content, content

### Agent行动循环

In [45]:
for i in range(5): # 设置最大循环次数
    print(f"--- 循环 {i+1} ---\n")
    
    # 3.1. 构建Prompt
    full_prompt = "\n".join(prompt_history)
    # 封装到messages
    messages = [
        {'role': 'system', 'content': AGENT_SYSTEM_PROMPT},
        {'role': 'user', 'content': full_prompt}
    ]
    print("历史记录", messages)
    # 3.2. 调用LLM进行思考
    think_output, output = get_qwen_output(messages, model, tokenizer)
    print(f"模型think:\n{think_output}\n")
    # 模型可能会输出多余的Thought-Action，需要截断
    match = re.search(r'(Thought:.*?Action:.*?)(?=\n\s*(?:Thought:|Action:|Observation:)|\Z)', output, re.DOTALL)
    if match:
        truncated = match.group(1).strip()
        if truncated != output.strip():
            output = truncated
            print("已截断多余的 Thought-Action 对")
    print(f"模型输出:\n{output}\n")
    prompt_history.append(output)
    
    # 3.3. 解析并执行行动
    action_match = re.search(r"Action: (.*)", output, re.DOTALL)
    if not action_match:
        observation = "错误: 未能解析到 Action 字段。请确保你的回复严格遵循 'Thought: ... Action: ...' 的格式。"
        observation_str = f"Observation: {observation}"
        print(f"{observation_str}\n" + "="*40)
        prompt_history.append(observation_str)
        continue
    action_str = action_match.group(1).strip()

    # if action_str.startswith("Finish"):
    if "Finish" in action_str:
        final_answer = re.search(r"Finish\[(.*)\]", action_str).group(1)
        print(f"任务完成，最终答案: {final_answer}")
        break
    
    tool_name = re.search(r"(\w+)\(", action_str).group(1)
    args_str = re.search(r"\((.*)\)", action_str).group(1)
    kwargs = dict(re.findall(r'(\w+)="([^"]*)"', args_str))

    if tool_name in available_tools:
        observation = available_tools[tool_name](**kwargs)
    else:
        observation = f"错误:未定义的工具 '{tool_name}'"

    # 3.4. 记录观察结果
    observation_str = f"Observation: {observation}"
    print(f"{observation_str}\n" + "="*40)
    prompt_history.append(observation_str)

--- 循环 1 ---

历史记录 [{'role': 'system', 'content': '\n你是一个智能旅行助手。你的任务是分析用户的请求，并使用可用工具一步步地解决问题。\n\n# 可用工具:\n- `get_weather(city: str)`: 查询指定城市的实时天气。\n- `get_attraction(city: str, weather: str)`: 根据城市和天气搜索推荐的旅游景点。\n\n# 输出格式要求:\n你的每次回复必须严格遵循以下格式，包含一对Thought和Action：\n\nThought: [你的思考过程和下一步计划]\nAction: [你要执行的具体行动]\n\nAction的格式必须是以下之一！：\n1. 调用工具：function_name(arg_name="arg_value")，例如：get_weather(city="北京") 。\n2. 结束任务：Finish[你的最终答案内容]。\n\n# 重要提示:\n- 每次只输出一对Thought-Action\n- Action必须在同一行，不要换行\n- 当收集到足够信息可以回答用户问题时，必须使用 Action: Finish[你的最终答案内容] 格式结束\n\n请开始吧！\n'}, {'role': 'user', 'content': '用户请求: 你好，请帮我查询一下今天苏州的天气，然后根据天气推荐一个合适的旅游景点。'}]
模型think:
<think>
好的，用户让我查询苏州今天的天气，然后根据天气推荐旅游景点。首先，我需要确认用户的需求。用户可能是在计划去苏州的旅行，所以需要知道天气情况来安排行程。接下来，我应该使用get_weather工具来获取苏州的实时天气数据。然后，使用get_attraction工具，传入苏州和查询到的天气信息，得到推荐的景点。

首先，我需要调用get_weather函数，参数是city="苏州"。然后，得到天气数据后，再调用get_attraction，传入city="苏州"和weather参数。在调用这两个工具后，应该能获取到具体的天气情况和景点推荐，最后将结果整理成用户友好的回答。
</think>

已截断多余的 Thought-Action 对
模型输出:
Thought: 调用get_weathe

In [84]:
del model
del tokenizer
CleanMemory()

**如果结果有问题，那就对了，下面是使用LLM正确输出样例**

```
用户输入: 你好，请帮我查询一下今天北京的天气，然后根据天气推荐一个合适的旅游景点。
========================================
--- 循环 1 ---

正在调用大语言模型...
大语言模型响应成功。
模型输出:
Thought: 首先需要获取北京今天的天气情况，之后再根据天气情况来推荐旅游景点。
Action: get_weather(city="北京")

Observation: 北京当前天气:Sunny，气温26摄氏度
========================================      
--- 循环 2 ---

正在调用大语言模型...
大语言模型响应成功。
模型输出:
Thought: 现在已经知道了北京今天的天气是晴朗且温度适中，接下来可以基于这个信息来推荐一个适合的旅游景点了。
Action: get_attraction(city="北京", weather="Sunny")

Observation: 北京在晴天最值得去的旅游景点是颐和园，因其美丽的湖景和古建筑。另一个推荐是长城，因其壮观的景观和历史意义。
========================================
--- 循环 3 ---

正在调用大语言模型...
大语言模型响应成功。
模型输出:
Thought: 已经获得了两个适合晴天游览的景点建议，现在可以根据这些信息给用户提供满意的答复。
Action: Finish[今天北京的天气是晴朗的，气温26摄氏度，非常适合外出游玩。我推荐您去颐和园欣赏美丽的湖景和古建筑，或者前往长城体验其壮观的景观和深厚的历史意义。希望您有一个愉快的旅行！]

任务完成，最终答案: 今天北京的天气是晴朗的，气温26摄氏度，非常适合外出游玩。我推荐您去颐和园欣赏美丽的湖景和古建筑，或者前往长城体验其壮观的景观和深厚的历史意义。希望您有一个愉快的旅行！
```

# 提示词工程

**核心定义**：如何用自然语言/结构化指令，精准地激发和引导 LLM 的推理能力与输出格式。它解决的是 “Agent 怎么想” 的问题。

## 智能体核心工作流（思维范式）

### ReAct和其实现

ReAct范式通过一种特殊的提示工程来引导模型，使其每一步的输出都遵循一个固定的轨迹：

- Thought (思考)： 这是智能体的“内心独白”。它会分析当前情况、分解任务、制定下一步计划，或者反思上一步的结果。
- Action (行动)： 这是智能体决定采取的具体动作，通常是调用一个外部工具，例如 Search['华为最新款手机']。
- Observation (观察)： 这是执行Action后从外部工具返回的结果，例如搜索结果的摘要或API的返回值。

智能体将不断重复这个 Thought -> Action -> Observation 的循环，将新的观察结果追加到历史记录中，形成一个不断增长的上下文，直到它在Thought中认为已经找到了最终答案，然后输出结果。

```mermaid
graph LR
    Q[用户问题] --> T1[Thought: 分析问题,</br>决定下一步行动]
    T1 --> A[Action: 调用工具/搜索/计算]
    A --> O[Observation: 观察返回结果]
    O --> J{得到最终答案?}
    J -- 否 --> T2[Thought: 基于观察再推理]
    T2 --> T1
    J -- 是 --> F[Final Answer]

#### 系统提示词

In [128]:
# ReAct 提示词模板
REACT_PROMPT_TEMPLATE = """
请注意，你是一个有能力调用外部工具的智能助手。

可用工具如下:
{tools}

**请严格按照以下格式进行回应!**:

Thought: 你的思考过程，用于分析问题、拆解任务和规划下一步行动。
Action: 你决定采取的行动，只能是以下格式中的一个:
- `{{tool}}[{{params}}]`:调用一个可用工具。
- `Finish[最终答案]`:当你认为已经获得最终答案时。
当你收集到足够的信息，能够回答用户的最终问题时，你必须在Action:字段后使用 Finish[最终答案] 来输出最终答案。

现在，请开始解决以下问题:
Question: {question}
History: {history}
"""

#### 工具与工具包

In [86]:
def search(query: str):
    print(f"🔍 正在进行网页搜索: {query}")
    return """截至2026年6月1日，英伟达最新发布的消费级桌面显卡是 GeForce RTX 50 系列（基于 Blackwell 架构），
    包括 RTX 5090、5080 等型号；专业/数据中心领域最新为 RTX PRO Blackwell（如 RTX PRO 6000）和 H200/H100 系
    列（H200 于2024年发布，仍属当前旗舰AI加速卡）。‌‌
    """

In [87]:
from typing import Dict, Any

class ToolExecutor:
    def __init__(self):
        self.tools: Dict[str, Dict[str, Any]] = {}

    def registerTool(self, name: str, description: str, func: callable):
        if name in self.tools:
            print(f"警告:工具 '{name}' 已存在，将被覆盖。")
        self.tools[name] = {"description": description, "func": func}
        print(f"工具 '{name}' 已注册。")

    def getTool(self, name: str) -> callable:
        return self.tools.get(name, {}).get("func")

    def getAvailableTools(self) -> str:
        return "\n".join([
            f"- {name}: {info['description']}" 
            for name, info in self.tools.items()
        ])

In [98]:
toolExecutor = ToolExecutor()
search_description = "一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。"
toolExecutor.registerTool("Search", search_description, search)
print("\n--- 可用的工具 ---")
print(toolExecutor.getAvailableTools())

tool_name = "Search"
tool_input = "英伟达最新GPU型号"
tool_function = toolExecutor.getTool(tool_name)
observation = tool_function(tool_input)
print(observation)

工具 'Search' 已注册。

--- 可用的工具 ---
- Search: 一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。
🔍 正在进行网页搜索: 英伟达最新GPU型号
截至2026年6月1日，英伟达最新发布的消费级桌面显卡是 GeForce RTX 50 系列（基于 Blackwell 架构），
    包括 RTX 5090、5080 等型号；专业/数据中心领域最新为 RTX PRO Blackwell（如 RTX PRO 6000）和 H200/H100 系
    列（H200 于2024年发布，仍属当前旗舰AI加速卡）。‌‌
    


#### 输出解析

In [116]:
def ReAct_parse_output(self, text: str):
    
    # 从Thought: 匹配到 Action: 或文本末尾
    thought_match = re.search(r"Thought:\s*(.*?)(?=\nAction:|$)", text, re.DOTALL)
    
    # 从Action: 匹配到文本末尾
    action_match = re.search(r"Action:\s*(.*?)$", text, re.DOTALL)
    
    thought = thought_match.group(1).strip() if thought_match else None
    action = action_match.group(1).strip() if action_match else None
    return thought, action

In [117]:
text="""Thought: 首先需要获取北京今天的天气情况，之后再根据天气情况来推荐旅游景点。
Action: Get_weather[北京]

Observation: 北京当前天气:Sunny，气温26摄氏度
"""
thought, action = ReAct_parse_output(None, text)
print("thought:", thought)
print("action:", action)

thought: 首先需要获取北京今天的天气情况，之后再根据天气情况来推荐旅游景点。
action: Get_weather[北京]

Observation: 北京当前天气:Sunny，气温26摄氏度


#### 工具解析

In [118]:
def ReAct_parse_action(self, action_text: str):

    # 从头匹配，先匹配字母数字下划线：get_weather，然后匹配 [，然后批匹配任意字符，最后匹配 ]
    # re.DOTALL :让 . 匹配包括换行符在内的所有字符
    match = re.match(r"(\w+)\[(.*)\]", action_text, re.DOTALL)
    if match:
        return match.group(1), match.group(2)
    return None, None

In [119]:
tool, params = ReAct_parse_action(None, action)
print("调用函数：",tool,"(",params,")")

调用函数： Get_weather ( 北京 )


#### ReActAgent简单实现

In [132]:
class ReActAgent:
    def __init__(self, tool_executor: ToolExecutor, model=None, max_steps=5):
        self.model = model,
        self.tool_executor = tool_executor
        self.max_steps = max_steps
        self.history = []
        
    def run(self, question: str):
        self.history = [] 
        current_step = 0
        model, tokenizer = load_model()
        while current_step < self.max_steps:
            current_step+=1
            print(f"--- 第 {current_step} 步 ---")
            
            # 准备提示词
            available_tools = self.tool_executor.getAvailableTools()
            history_str = "\n".join(self.history)
            prompt = REACT_PROMPT_TEMPLATE.format(
                tools=available_tools,
                question=question,
                history=history_str
            )
            # print("提示词：",prompt,"\n")
            
            # 模型推理
            messages = [{"role": "user", "content": prompt}]
            think_output, output = get_qwen_output(messages, model, tokenizer)
            # print(think_output, "\n" ,output)
            # 输出解析
            thought, action = self.ReAct_parse_output(output)
            print("思考过程：",thought) if thought else None
            if not action:
                print(output)
                print(thought, action)
                break
                
            if action.startswith("Finish"):
                # 如果是Finish指令，提取最终答案并结束
                final_answer = re.match(r"Finish\[(.*)\]", action).group(1)
                print(f"🎉 最终答案: {final_answer}")
                return final_answer

            # 工具解析
            tool, params = self.ReAct_parse_action(action)
            if not tool or not params:
                print(output)
                print(tool, params)
                break

            print(f"🎬 行动: {tool}[{params}]")
            
            tool_function = self.tool_executor.getTool(tool_name)
            if not tool_function:
                observation = f"错误:未找到名为 '{tool}' 的工具。"
            else:
                observation = tool_function(params) # 调用真实工具
                
            print(f"👀 观察: {observation}")
            # 将本轮的Action和Observation添加到历史记录中
            self.history.append(f"Action: {action}")
            self.history.append(f"Observation: {observation}")

        # 循环结束
        del model
        del tokenizer
        CleanMemory()
        print("已达到最大步数，流程终止。")

In [133]:
ReActAgent.ReAct_parse_output = ReAct_parse_output
ReActAgent.ReAct_parse_action = ReAct_parse_action

In [134]:
agent = ReActAgent(toolExecutor)
agent.run("英伟达最新的GPU型号是什么？")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

--- 第 1 步 ---
思考过程： 英伟达（NVIDIA）最新推出的GPU型号是A100，这是2023年发布的，目前在市场中广泛使用。
🎬 行动: Search[NVIDIA最新GPU型号]  
Finish[NVIDIA最新GPU型号是A100]
🔍 正在进行网页搜索: NVIDIA最新GPU型号]  
Finish[NVIDIA最新GPU型号是A100
👀 观察: 截至2026年6月1日，英伟达最新发布的消费级桌面显卡是 GeForce RTX 50 系列（基于 Blackwell 架构），
    包括 RTX 5090、5080 等型号；专业/数据中心领域最新为 RTX PRO Blackwell（如 RTX PRO 6000）和 H200/H100 系
    列（H200 于2024年发布，仍属当前旗舰AI加速卡）。‌‌
    
--- 第 2 步 ---
Finish[NVIDIA最新GPU型号是A100]
None None
已达到最大步数，流程终止。


**显然0.6B的小模型还是听不懂提示词，正确结果如下**

```
问题：华为最新的手机是哪个？
--- 第 1 步 ---
Thought: 要回答这个问题，我需要查找华为最新发布的手机型号及其主要特点。这些信息可能在我的现有知识库之外，因此需要使用搜索引擎来获取最新数据。
Action: Search[华为最新手机型号及主要卖点]
🤔 思考: 要回答这个问题，我需要查找华为最新发布的手机型号及其主要特点。这些信息可能在我的现有知识库之外，因此需要使用搜索引擎来获取最新数据。
🎬 行动: Search[华为最新手机型号及主要卖点]
🔍 正在执行 [SerpApi] 网页搜索: 华为最新手机型号及主要卖点
👀 观察: [1] 华为手机- 华为官网
智能手机 ; Mate 系列. 非凡旗舰 · HUAWEI Mate XTs. 非凡大师 ; Pura 系列. 先锋影像 · HUAWEI Pura 80 Pro+ ; Pocket 系列. 美学新篇. HUAWEI Pocket 2 ; nova 系列. 专业人像.

[2] 2025年华为手机哪一款性价比高？华为手机推荐与市场分析 ...
现在华为手机最大的卖点只剩下鸿蒙HarmonyOS系统，以及饱受争议的品牌信仰。 这里推荐目前值得入手的几款华为系列手机，根据不同预算自行选择:. 华为目前最受欢迎，也是搭载 ...

[3] 2025年华为新款手机哪个性价比高？10款华为新款手机推荐
选华为主要还是要推荐高端手机，Mate 70和Pura 70系列是最新发布的旗舰机型。 HUAWEI Mate 70. 优点是，拍照配置依旧顶级，全焦段覆盖，适合专业摄影，做工出色，户外抗摔 ...

--- 第 2 步 ---
Thought: 根据搜索结果，华为最新发布的旗舰机型包括Mate 70和Pura 80 Pro+。为了确定最新型号及其主要卖点，我将重点放在这些信息上。从提供的链接来看，Mate 70系列和Pura 80 Pro+都是近期发布的产品，但具体哪一个是“最新”还需要进一步确认。同时，我可以从这些信息中提取出它们的主要
卖点。
Action: Finish[根据最新信息，华为的最新手机可能是HUAWEI Pura 80 Pro+或HUAWEI Mate 70。其中，HUAWEI Mate 70的主要卖点包括顶级的拍照配置，全焦段覆盖，适合专业摄影，做工出色，并且具有良好的户外抗摔性能。而HUAWEI Pura 80 Pro+则强调了先锋影像技术。]
🤔 思考: 根据搜索结果，华为最新发布的旗舰机型包括Mate 70和Pura 80 Pro+。为了确定最新型号及其主要卖点，我将重点放在这些信息上。从提供的链接来看，Mate 70系列和Pura 80 Pro+都是近期发布的产品，但具体哪一个是“最新”还需要进一步确认。同时，我可以从这些信息中提取出它们的主要 
卖点。
🎉 最终答案: 根据最新信息，华为的最新手机可能是HUAWEI Pura 80 Pro+或HUAWEI Mate 70。其中，HUAWEI Mate 70的主要卖点包括顶级的拍照配置，全焦段覆盖，适合专业摄影，做工出色，并且具有良好的户外抗摔性能。而HUAWEI Pura 80 Pro+则强调了先锋影像技术。

### Plan-and-Solve

顾名思义，这种范式将任务处理明确地分为两个阶段：先规划 (Plan)，后执行 (Solve)。

- 规划阶段 (Planning Phase)： 首先，智能体会接收用户的完整问题。它的第一个任务不是直接去解决问题或调用工具，而是将问题分解，并制定出一个清晰、分步骤的行动计划。这个计划本身就是一次大语言模型的调用产物。
- 执行阶段 (Solving Phase)： 在获得完整的计划后，智能体进入执行阶段。它会严格按照计划中的步骤，逐一执行。每一步的执行都可能是一次独立的 LLM 调用，或者是对上一步结果的加工处理，直到计划中的所有步骤都完成，最终得出答案。

```mermaid
graph LR
    Q[复杂问题] --> P[Planner: 把问题拆解成<br/>子步骤]
    P --> S1[Step 1:执行子任务1]
    S1 --> S2[Step 2:执行子任务2]
    S2 --> SN[...]
    SN --> OUT[综合输出最终答案]

### Reflection

为智能体引入一种事后（post-hoc）的自我校正循环，使其能够像人类一样，审视自己的工作，发现不足，并进行迭代优化。

其核心工作流程可以概括为一个简洁的三步循环：执行 -> 反思 -> 优化。

- 执行 (Execution)：首先，智能体使用我们熟悉的方法（如 ReAct 或 Plan-and-Solve）尝试完成任务，生成一个初步的解决方案或行动轨迹。这可以看作是“初稿”。
- 反思 (Reflection)：接着，智能体进入反思阶段。它会调用一个独立的、或者带有特殊提示词的大语言模型实例，来扮演一个“评审员”的角色。这个“评审员”会审视第一步生成的“初稿”，并从多个维度进行评估，例如：
  - 事实性错误：是否存在与常识或已知事实相悖的内容？
  - 逻辑漏洞：推理过程是否存在不连贯或矛盾之处？
  - 效率问题：是否有更直接、更简洁的路径来完成任务？
  - 遗漏信息：是否忽略了问题的某些关键约束或方面？ 根据评估，它会生成一段结构化的反馈 (Feedback)，指出具体的问题所在和改进建议。
- 优化 (Refinement)：最后，智能体将“初稿”和“反馈”作为新的上下文，再次调用大语言模型，要求它根据反馈内容对初稿进行修正，生成一个更完善的“修订稿”。

**可以用同一个模型切换角色，节省成本；也可以是两个不同模型**

```mermaid
graph LR
    Q[任务输入] --> G[执行: 生成初版答案]
    G --> R[反思: 自我评估/批评]
    R --> J{质量达标?}
    J -- 否 --> M[优化: 根据反馈修改/重写]
    M --> R
    J -- 是 --> F[输出最终答案]

### 总结

 |   | ReAct   | 	Plan-and-Solve  | 	Reflection |
 | ---- | ---- | ---- |----|
 | 核心思路 | 	边想边做,交替进行  | 	先全盘规划,再逐步执行 | 	做完了再回头审视、修正 |
 | 关键动作 | 	Thought → Action → Observation	 | Plan → Steps → Output | 	Produce → Critique → Refine |
 | 适合场景 | 	需要调用工具/搜索/外部信息 | 	多步推理、数学、应用题 | 	需要高质量输出、自纠错 |

## 智能体搭建平台

**Coze**：https://www.coze.cn/

- 核心定位：由字节跳动推出的 Coze[1]，主打零代码/低代码的 Agent 的构建体验，让不具备编程背景的用户也能轻松创造。
- 特点分析：Coze 拥有极其友好的可视化界面，用户可以像搭建乐高积木一样，通过拖拽插件、配置知识库和设定工作流来创建智能体。其内置了极为丰富的插件库，并支持一键发布到抖音、飞书、微信公众号等多个主流平台，极大地简化了分发流程。
- 适用人群：**非技术用户**，AI 应用的入门用户、产品经理、运营人员，以及希望快速将创意变为可交互产品的个人创作者。

**Dify**

- 核心定位：Dify 是一个开源的、功能全面的 LLM 应用开发与运营平台[2]，旨在为开发者提供从原型构建到生产部署的一站式解决方案。
- 特点分析：它融合了后端服务和模型运营的理念，支持 Agent 工作流、RAG Pipeline、数据标注与微调等多种能力。对于追求专业、稳定、可扩展的企业级应用而言，Dify 提供了坚实的基础。
- 适用人群：**开发者**，有一定技术背景的开发者、需要构建可扩展的企业级 AI 应用的团队。

**n8n**

- 核心定位：n8n 本质上是一个开源工作流自动化工具[3]，而非纯粹的 LLM 平台。近年来，它积极集成了 AI 能力。
- 特点分析：n8n 的强项在于“连接”。它拥有数百个预置的节点，可以轻松地将各类 SaaS 服务、数据库、API 连接成复杂的自动化业务流程。你可以在这个流程中嵌入 LLM 节点，使其成为整个自动化链路中的一环。虽然在 LLM 功能的专一度上不如前两者，但其通用自动化能力是独一无二的。不过，其学习曲线也相对陡峭。
- 适用人群：**AI只是其中一个节点**，需要将 AI 能力深度整合进现有业务流程、实现高度定制化自动化的开发者和企业。

**通用ai智能体**

- MiniMax Agent：https://agent.minimaxi.com/
- 智谱 Agent：https://chatglm.cn/
- Trae SOLO：https://solo.trae.cn/

# 上下文工程

**核心定义**：如何在有限的 Token 窗口内，动态地装载、裁剪、整合 LLM 决策所需的信息。它解决的是 “Agent 知道什么” 的问题。这是目前 Agent 工程中最难、最核心的部分。

## 记忆系统

人类记忆系统

```mermaid
graph LR
    env[环境输入]
    
    subgraph SM[感觉记忆]
        direction LR
        vis[视觉]
        hear[听觉]
        touch[触觉]
        more[......]
    end
    
    subgraph STM[短时记忆]
        direction TB
        stm[短时记忆]
        stm --> stm
    end
    
    ltm[长时记忆]
    
    f1[遗忘]
    f2[遗忘<br/>因衰退或干扰]
    f3[遗忘<br/>因干扰或提取失败]
    
    env ==> SM
    SM ==> STM
    SM --> f1
    
    STM ==>|存储| ltm
    ltm ==>|提取| STM
    
    STM --> f2
    ltm --> f3
    

    classDef envStyle fill:#a7f3d0,stroke:#059669,color:#000
    classDef memStyle fill:#fed7aa,stroke:#ea580c,color:#000
    classDef forgetStyle fill:#e5e7eb,stroke:#6b7280,color:#000
    
    class env envStyle
    class vis,hear,touch,more,stm,ltm,reh memStyle
    class f1,f2,f3 forgetStyle

**记忆系统要解决的问题**：
- 上下文丢失：在长对话中，早期的重要信息可能会因为上下文窗口限制而丢失
- 个性化缺失：Agent无法记住用户的偏好、习惯或特定需求
- 学习能力受限：无法从过往的成功或失败经验中学习改进
- 一致性问题：在多轮对话中可能出现前后矛盾的回答
- 知识时效性：大模型的训练数据有时间截止点，无法获取最新信息
- 专业领域知识：通用模型在特定领域的深度知识可能不足
- 事实准确性：通过检索验证，减少模型的幻觉问题
- 可解释性：提供信息来源，增强回答的可信度

### 记忆系统工作流程

```mermaid
graph LR
    subgraph EP[外部信息处理]
        SI[感知输入<br/>Sensory Input] --> EN[编码<br/>Encoding]
    end
    
    subgraph MC[记忆系统核心]
        ST{存储<br/>Storage}
        CO[整合<br/>Consolidation]
        FO[遗忘<br/>Forgetting]
        RE[检索<br/>Retrieval]
    end
    
    subgraph OUT[输出]
        RB[回忆与行为输出<br/>Recall & Behavior]
    end
    
    EN --> ST
    ST -->|信息巩固与强化| CO
    ST -->|信息丢失| FO
    ST -->|信息提取| RE
    RE --> RB

    classDef purpleStyle fill:#e9d5ff,stroke:#7c3aed,color:#000
    classDef blueStyle fill:#bfdbfe,stroke:#2563eb,color:#000
    classDef orangeStyle fill:#fed7aa,stroke:#ea580c,color:#000
    classDef greenStyle fill:#bbf7d0,stroke:#16a34a,color:#000
    classDef redStyle fill:#fecaca,stroke:#dc2626,color:#000
    
    class SI,RB purpleStyle
    class EN,RE blueStyle
    class ST orangeStyle
    class CO greenStyle
    class FO redStyle

### 轻量级记忆系统演示

In [8]:
# !pip install chromadb
!pip show chromadb

Name: chromadb
Version: 1.5.9
Summary: Chroma.
Home-page: https://github.com/chroma-core/chroma
Author: 
Author-email: Jeff Huber <jeff@trychroma.com>, Anton Troynikov <anton@trychroma.com>
License: 
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: bcrypt, build, grpcio, httpx, importlib-resources, jsonschema, kubernetes, mmh3, numpy, onnxruntime, opentelemetry-api, opentelemetry-exporter-otlp-proto-grpc, opentelemetry-sdk, orjson, overrides, pybase64, pydantic, pydantic-settings, pypika, pyyaml, rich, tenacity, tokenizers, tqdm, typer, typing-extensions, uvicorn
Required-by: 


In [1]:
import time
import chromadb
from chromadb.utils import embedding_functions

In [2]:
# ========== 配置 ==========
EMBEDDING_MODEL = "./BAAI/bge-small-zh-v1.5"  # 本地路径
PERSIST_DIR = "./memory_db"
COLLECTION_NAME = "memories"

# ========== 初始化(自动加载 bge,自动建库)==========
client = chromadb.PersistentClient(path=PERSIST_DIR)           # 创建数据库
ef = embedding_functions.SentenceTransformerEmbeddingFunction( # 加载嵌入器
    model_name=EMBEDDING_MODEL
)
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,                     # 数据库中的表名
    embedding_function=ef,                    # 数据转向量的方式
    metadata={"hnsw:space": "cosine"}         # 向量相似度计算方式：余弦距离
)

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ./BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


存储

In [3]:
def remember(text: str, importance: float = 0.5, source: str = "chat", user_id: str = "default"):
    """存一条记忆,自动加 metadata"""
    mid = f"mem_{int(time.time() * 1_000_000)}"   # 记忆id
    collection.add(
        documents=[text],                  # 记忆内容
        metadatas=[{ 
            "user_id": user_id,            # 用户id
            "importance": importance,      # 重要性 0~1
            "created_at": time.time(),     # 创建时间
            "last_access": time.time(),    # 上次访问时间 
            "access_count": 0,             # 访问次数
            "source": source               # 来源
        }],
        ids=[mid]                          # 记忆id
    )
    print(f"  💾保存 [{importance}] {text}")

检索

In [4]:
def recall(query: str, k: int = 3, user_id: str = "default"):
    raw = collection.query(                             # 查找user_id 的 topk个 与 query相关的记忆
        query_texts=[query],
        n_results=k,
        where={"user_id": user_id}
    )
    
    if not raw["documents"] or not raw["documents"][0]: # 查找失败
        return []

    docs, ids, metas, dists = (raw["documents"][0], raw["ids"][0], raw["metadatas"][0], raw["distances"][0])

    scored = []
    for doc, mid, meta, dist in zip(docs, ids, metas, dists):
        sim = 1 - dist                                  # 相似度 = 1 - 余弦距离
        age = time.time() - meta["last_access"]         # 保存多久了
        decay = 0.5 ** (age / (7 * 86400))              # 7 天半衰期
        
        score = sim * decay * (0.8 + meta["importance"] * 0.4) # 计算该记忆的分数
        scored.append((score, doc, mid, meta, sim))

    scored.sort(key=lambda x: -x[0])                    # -x[0] 相当于取(score, doc, mid, meta, sim)中的score，并转为负值
    top = scored[:k]                                    # 取前 k个

    # 强化:被访问过的记忆下次更容易被找到
    # for _, _, mid, meta, _ in top:
    #     collection.update(
    #         ids=[mid],                                # 更新 mid 的数据
    #         metadatas=[{**meta,                       # {**meta, "key": value} 将后面的key对应的值 替换 meta原有key对应的值
    #                     "last_access": time.time(),   # 更新上次访问时间 
    #                     "access_count": meta["access_count"] + 1}] # 访问次数+1
    #     )
    # 建议使用下方的批量更新，只需要和数据库通信 1 次
    batch_ids = []
    batch_metas = []
    for _, _, mid, meta, _ in top:
        batch_ids.append(mid)
        batch_metas.append({**meta, 
                            "last_access": time.time(), 
                            "access_count": meta["access_count"] + 1})
    if batch_ids:                                        # 确保列表不为空
        collection.update(
            ids=batch_ids,                               # 传入包含多个 ID 的列表
            metadatas=batch_metas)                       # 传入包含多个字典的列表
        
    return [(doc, meta, score) for score, doc, _, meta, _ in top]

整合（合并相似记忆）

In [5]:
def consolidate(user_id: str = "default", threshold: float = 0.8):
    all_mems = collection.get(where={"user_id": user_id})                   # 获取所有记忆
    if not all_mems["documents"] or len(all_mems["documents"]) < 2:         # 记忆太少就返回
        print("  ✨ 记忆太少,无需整合")
        return 0

    merged = 0
    used = set()

    for i, doc in enumerate(all_mems["documents"]):
        if i in used:
            continue
        neighbors = collection.query(                                       # 找这条记忆的最相似的5条
            query_texts=[doc], 
            n_results=5,
            where={"user_id": user_id}
        )
        
        cluster_idx = [i]
        if neighbors["ids"] and neighbors["ids"][0]:                        # 如果有相似的
            for nb_id, dist in zip(neighbors["ids"][0][1:], neighbors["distances"][0][1:]):  # [1:] 的意思是：去掉列表的第 0个元素（也就是他自己），只保留后面的元素
                if 1 - dist < threshold:                                    # 相似度小于阈值就遍历下一条
                    continue
                    
                try:
                    nb_idx = all_mems["ids"].index(nb_id)
                    if nb_idx not in used and nb_idx != i:
                        cluster_idx.append(nb_idx)                          # 加入簇
                        used.add(nb_idx)                                    # 标记被用了
                except ValueError:
                    pass

        if len(cluster_idx) > 1:                                            # 如果簇中有2个及以上数据了
            # 合并:保留最重要的,重要性取 max,访问次数累加
            metas_in = [all_mems["metadatas"][k] for k in cluster_idx]
            # best_k = max(cluster_idx, key=lambda k: metas_in[cluster_idx.index(k)]["importance"])
            best_k = max(cluster_idx, key=lambda k: all_mems["metadatas"][k]["importance"])       # 取 重要性最高 的那条记忆的 索引
            best_meta = all_mems["metadatas"][best_k].copy()
            best_meta["importance"] = min(1.0, max(m["importance"] for m in metas_in) + 0.1)      # 取最重要的，并+0.1
            best_meta["access_count"] = sum(m["access_count"] for m in metas_in)                  # 更新访问次数为合并的记忆和

            # 删掉其他,保留 best
            for k in cluster_idx:
                if k != best_k:
                    collection.delete(ids=[all_mems["ids"][k]])
                    merged += 1
                    
            collection.update(
                ids=[all_mems["ids"][best_k]],
                metadatas=[best_meta]
            )
            print(f"  🔗 整合 {len(cluster_idx)} 条 → \"{all_mems['documents'][best_k][:30]}...\"")

    if merged == 0:
        print("  ✨ 没有需要整合的")
    return merged                        # 返回因合并而删除的数量

遗忘（清理低分记忆）

In [6]:
def forget(user_id: str = "default", keep_top: int = 20):
    """基于 importance × 时间衰减 × 访问频率,淘汰最差的"""
    all_mems = collection.get(where={"user_id": user_id})
    if not all_mems["documents"] or len(all_mems["documents"]) <= keep_top:
        print(f"  ✨ 记忆 {len(all_mems['documents']) if all_mems['documents'] else 0} 条,无需遗忘")
        return 0

    scored = []
    for mid, meta in zip(all_mems["ids"], all_mems["metadatas"]):
        age = time.time() - meta["last_access"]
        decay = 0.5 ** (age / (7 * 86400))
        score = meta["importance"] * decay * (1 + meta["access_count"] * 0.2)
        scored.append((score, mid))

    scored.sort()  # 升序:差的在前
    to_remove = len(all_mems["documents"]) - keep_top
    for score, mid in scored[:to_remove]:
        # 拿原文用来打印
        info = collection.get(ids=[mid])
        doc = info["documents"][0] if info["documents"] else "?"
        collection.delete(ids=[mid])
        print(f"  🗑️ 遗忘: \"{doc[:30]}...\"(分={score:.2f})")
    return to_remove

打印所有记忆

In [7]:
def list_all(user_id: str = "default"):
    all_mems = collection.get(where={"user_id": user_id})
    n = len(all_mems["documents"]) if all_mems["documents"] else 0
    print(f"\n📊 {user_id} 的记忆库({n} 条):\n")
    if not all_mems["documents"]:
        print("  📭 空空如也")
        return
    for i, (doc, meta) in enumerate(zip(all_mems["documents"], all_mems["metadatas"]), 1):
        print(f"  {i}. [imp={meta['importance']:.1f}|×{meta['access_count']}] {doc}")

In [8]:
def demo():
    print("=" * 60)
    print("🧠 极简记忆系统演示")
    print("=" * 60)

    user = "zhangsan"

    # 1. 存储
    print("\n📥 Step 1: 存储 5 条记忆\n")
    remember("用户叫张三,Python 开发者", importance=0.9, user_id=user)
    remember("用户喜欢简洁代码,讨厌冗余", importance=0.7, user_id=user)
    remember("今天北京天气晴,25 度", importance=0.2, user_id=user)
    remember("用户姓张,是 Python 程序员", importance=0.85, user_id=user)  # 跟第1条相似
    remember("用户偏好简洁代码风格", importance=0.75, user_id=user)  # 跟第2条相似
    list_all(user)

    # 2. 整合
    print("\n🔗 Step 2: 整合相似记忆\n")
    consolidate(user, threshold=0.8)
    list_all(user)

    # 3. 检索
    print("\n🔍 Step 3: 检索 \"用户的编程偏好\"\n")
    for doc, meta, score in recall("用户的编程偏好", k=3, user_id=user):
        print(f"  ✅ [分={score:.2f}|×{meta['access_count']}] {doc}")

    # 4. 遗忘(模拟时间流逝)
    print("\n⏰ Step 4: 模拟 30 天后,触发遗忘\n")
    all_mems = collection.get(where={"user_id": user})
    for mid, meta in zip(all_mems["ids"], all_mems["metadatas"]):
        if meta["importance"] < 0.5:
            collection.update(
                ids=[mid],
                metadatas=[{**meta, "last_access": time.time() - 30 * 86400}]
            )
    forget(user, keep_top=3)
    list_all(user)

    # 5. 再检索
    print("\n🔍 Step 5: 30 天后检索\n")
    for doc, meta, score in recall("用户的编程偏好", k=3, user_id=user):
        print(f"  ✅ [分={score:.2f}] {doc}")

demo()

🧠 极简记忆系统演示

📥 Step 1: 存储 5 条记忆

  💾保存 [0.9] 用户叫张三,Python 开发者
  💾保存 [0.7] 用户喜欢简洁代码,讨厌冗余
  💾保存 [0.2] 今天北京天气晴,25 度
  💾保存 [0.85] 用户姓张,是 Python 程序员
  💾保存 [0.75] 用户偏好简洁代码风格

📊 zhangsan 的记忆库(5 条):

  1. [imp=0.9|×0] 用户叫张三,Python 开发者
  2. [imp=0.7|×0] 用户喜欢简洁代码,讨厌冗余
  3. [imp=0.2|×0] 今天北京天气晴,25 度
  4. [imp=0.8|×0] 用户姓张,是 Python 程序员
  5. [imp=0.8|×0] 用户偏好简洁代码风格

🔗 Step 2: 整合相似记忆

  🔗 整合 2 条 → "用户叫张三,Python 开发者..."
  🔗 整合 2 条 → "用户偏好简洁代码风格..."

📊 zhangsan 的记忆库(3 条):

  1. [imp=1.0|×0] 用户叫张三,Python 开发者
  2. [imp=0.2|×0] 今天北京天气晴,25 度
  3. [imp=0.8|×0] 用户偏好简洁代码风格

🔍 Step 3: 检索 "用户的编程偏好"

  ✅ [分=0.88|×0] 用户偏好简洁代码风格
  ✅ [分=0.72|×0] 用户叫张三,Python 开发者
  ✅ [分=0.31|×0] 今天北京天气晴,25 度

⏰ Step 4: 模拟 30 天后,触发遗忘

  ✨ 记忆 3 条,无需遗忘

📊 zhangsan 的记忆库(3 条):

  1. [imp=1.0|×1] 用户叫张三,Python 开发者
  2. [imp=0.2|×1] 今天北京天气晴,25 度
  3. [imp=0.8|×1] 用户偏好简洁代码风格

🔍 Step 5: 30 天后检索

  ✅ [分=0.88] 用户偏好简洁代码风格
  ✅ [分=0.72] 用户叫张三,Python 开发者
  ✅ [分=0.02] 今天北京天气晴,25 度


删除数据库

In [9]:
import shutil
shutil.rmtree(PERSIST_DIR, ignore_errors=True) 

### 图记忆系统 🚧

### 多模态记忆 🚧

## RAG

RAG（Retrieval-Augmented Generation）是结合检索和生成的AI框架，让Agent能够从外部知识库检索相关信息，结合这些信息生成更准确的回答
- 检索器：从知识库中查找相关文档
- 生成器：基于检索结果和问题生成回答

```mermaid
sequenceDiagram
    autonumber
    actor U as 👤 用户
    participant L as 📄 Loader
    participant SP as ✂️ Splitter
    participant EM as 🔢 Embedder
    participant DB as 🗄️ Vector DB
    participant RT as 🔍 Retriever
    participant LLM as 🧠 LLM

    Note over U,DB: 📥 阶段一:PDF 入库
    U->>L: 指定 PDF 路径
    L->>L: 按页解析
    L-->>SP: Document 列表
    SP->>SP: 切分 chunk_size=500
    SP-->>EM: 切好的 chunks
    EM->>EM: chunks → 向量
    EM->>DB: 存向量 + 原文
    DB-->>U: ✅ 入库完成

    Note over U,LLM: 🔍 阶段二:RAG 问答
    U->>RT: 提问
    RT->>EM: 问题 → 向量
    EM-->>RT: 问题向量
    RT->>DB: 相似度搜索
    DB-->>RT: top-k 相关 chunks
    RT->>LLM: 拼 prompt<br/>问题 + chunks
    LLM->>LLM: 生成答案
    LLM-->>U: 🎉 最终答案

#### RAG检索

In [27]:
# !pip install langchain langchain-community langchain-text-splitters langchain-openai chromadb pypdf sentence-transformers
!pip show langchain langchain-community langchain-text-splitters langchain-openai chromadb pypdf sentence-transformers

Name: langchain
Version: 1.3.2
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-community
Version: 0.4.2
Summary: Community contributed LangChain integrations.
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: aiohttp, httpx-sse, langchain-classic, langchain-core, langsmith, numpy, pydantic-settings, pyyaml, requests, sqlalchemy, tenacity
Required-by: 
---
Name: langchain-text-splitters
Version: 1.1.2
Summary: LangChain text splitting utilities
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: langchain-core
Required-by: langchain-cl

导入所需的库

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings, HuggingFaceBgeEmbeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.embeddings import ModelScopeEmbeddings
from langchain_community.vectorstores import Chroma

**加载pdf**

In [3]:
loader = PyPDFLoader("./data/Agent/Happy-LLM-0727.pdf")
documents = loader.load()                               # 每页一个 Document 对象
print(f"📄 加载完成,共 {len(documents)} 页")

📄 加载完成,共 171 页


**切分文本**

In [4]:
# RecursiveCharacterTextSplitter 是官方推荐的"递归切分器"
# 优先按段落 → 句子 → 词切,保持语义完整

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,                                     # 每块最大 500 字符
    chunk_overlap=50,                                   # 相邻块重叠 50 字符,避免切断上下文
    separators=["\n\n", "\n", "。", "!", "?", " ", ""]  # 中文友好分隔符
)
splits = text_splitter.split_documents(documents)
print(f"✂️  切分完成,共 {len(splits)} 个块")

✂️  切分完成,共 559 个块


**Embedding嵌入器**

In [74]:
!modelscope download --model BAAI/bge-small-zh-v1.5 --local_dir ./BAAI/bge-small-zh-v1.5


 _   .-')                _ .-') _     ('-.             .-')                              _ (`-.    ('-.
( '.( OO )_             ( (  OO) )  _(  OO)           ( OO ).                           ( (OO  ) _(  OO)
 ,--.   ,--.).-'),-----. \     .'_ (,------.,--.     (_)---\_)   .-----.  .-'),-----.  _.`     \(,------.
 |   `.'   |( OO'  .-.  ',`'--..._) |  .---'|  |.-') /    _ |   '  .--./ ( OO'  .-.  '(__...--'' |  .---'
 |         |/   |  | |  ||  |  \  ' |  |    |  | OO )\  :` `.   |  |('-. /   |  | |  | |  /  | | |  |
 |  |'.'|  |\_) |  |\|  ||  |   ' |(|  '--. |  |`-' | '..`''.) /_) |OO  )\_) |  |\|  | |  |_.' |(|  '--.
 |  |   |  |  \ |  | |  ||  |   / : |  .--'(|  '---.'.-._)   \ ||  |`-'|   \ |  | |  | |  .___.' |  .--'
 |  |   |  |   `'  '-'  '|  '--'  / |  `---.|      | \       /(_'  '--'\    `'  '-'  ' |  |      |  `---.
 `--'   `--'     `-----' `-------'  `------'`------'  `-----'    `-----'      `-----'  `--'      `------'


Successfully Downloaded from model BAAI/bge-small-zh

In [6]:
embedding = HuggingFaceBgeEmbeddings(model_name="./BAAI/bge-small-zh-v1.5")

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ./BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
vector = embedding.embed_query("什么是机器学习")
print(f"✅ 成功！向量维度: {len(vector)}")

✅ 成功！向量维度: 512


**存入Chroma向量数据库**

In [8]:
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embedding,
    # persist_directory="./data/Agent/chroma_db"          # 持久化目录,下次直接复用，  也可以只加载在内存

)
print(f"💾 入库完成,数据已存到 ./data/Agent/chroma_db")

💾 入库完成,数据已存到 ./data/Agent/chroma_db


删除数据库

In [9]:
# import shutil
# shutil.rmtree("./data/Agent/chroma_db", ignore_errors=True) 

**相似度检索**

In [10]:
query = "Agent是什么?"                          # 替换成你的问题
docs = vectorstore.similarity_search(query, k=3)        # 返回最相关的 3 块
print(f"\n🔍 查询: {query}")
print(f"📚 检索到 {len(docs)} 个相关块:\n")
for i, doc in enumerate(docs, 1):
    print(f"--- 块 {i} (来源: 第 {doc.metadata.get('page', '?') + 1} 页) ---")
    print(doc.page_content[:200])                       # 打印前 200 字


🔍 查询: Agent是什么?
📚 检索到 3 个相关块:

--- 块 1 (来源: 第 164 页) ---
个公司组织结构来完成商业策划。AutoGen、ChatDev等框架⽀持这类系统的构建。
探索与学习型Agent（Exploration & Learning Agents）：
特点： 这类Agent不仅执⾏任务，还能在与环境的交互中主动学习新知识、新技能或优化⾃身策略，类似于强
化学习中的Agent概念。
⼯作⽅式： 可能包含更复杂的记忆和反思机制，能够根据成功或失败的经验调整未来的规划和⾏动。

--- 块 2 (来源: 第 168 页) ---
Agent 的⼯作流程如下：
1. 接收⽤户输⼊。
2. 调⽤⼤模型（如 Qwen），并告知其可⽤的⼯具及其 Schema。
3. 如果模型决定调⽤⼯具，Agent 会解析请求，执⾏相应的 Python 函数。
4. Agent 将⼯具的执⾏结果返回给模型。
5. 模型根据⼯具结果⽣成最终回复。
6. Agent 将最终回复返回给⽤户。
如图7.9所示，Agent 调⽤⼯具流程：
图7.9 Age
--- 块 3 (来源: 第 162 页) ---
注：7.2 章节的所有代码均可在 Happy-LLM Chapter7 RAG 中找到。
7.3 Agent  
7.3.1 什么是 LLM Agent？  
简单来说，⼤模型Agent是⼀个以LLM为核⼼“⼤脑”，并赋予其⾃主规划、记忆和使⽤⼯具能⼒的系统。 它不再仅
仅是被动地响应⽤户的提示（Prompt），⽽是能够：
1. 理解⽬标（Goal Understanding）： 接收⼀个相对复杂


#### RAG增强生成

In [26]:
# 在简单写一个Agent里的函数
# model, tokenizer = load_model()
# think_output, output = get_qwen_output(messages, model, tokenizer)

**将我们本地的qwen3-0.6b接入langchain**

In [14]:
from langchain_core.language_models.llms import LLM
from typing import Optional, List, Any

# 编写 LangChain 适配器
class Qwen3LLM(LLM):
    model: Any
    tokenizer: Any

    @property
    def _llm_type(self) -> str:
        return "qwen3"

    def _call(self, prompt: str, stop: Optional[List[str]] = None, **kwargs) -> str:
        messages = [{"role": "user", "content": prompt}]
        think_output, output = get_qwen_output(messages, self.model, self.tokenizer)      # 调用我们的推理函数
        print(f"\n🧠 [模型思考]: {think_output}\n")                                       # 如果需要看思考过程，可以打印一下
        return output                                                                     # 只把最终输出返回给 RAG 链

In [15]:
model, tokenizer = load_model()
qwen3_06b_llm = Qwen3LLM(model=model, tokenizer=tokenizer)
print("✅ 本地 Qwen3-0.6B 已成功接入 LangChain！")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

✅ 本地 Qwen3-0.6B 已成功接入 LangChain！


In [17]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

**构建RAG链**

LCEL写法，全称是 LangChain Expression Language（LangChain 表达式语言）
```
# 像在搭积木，每一步清清楚楚
chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
```

In [19]:
template = """已知信息：
{context}

请根据上面的已知信息回答问题。
问题：{input}
回答："""

prompt = PromptTemplate.from_template(template)

# 构建 Retriever（从向量库检索 Top 3）
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 准备一个辅助函数：把检索到的 Document 对象列表合并成纯字符串
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
    
rag_chain = (
    # 第一步：构造字典，准备填坑的数据
    {
        "context": retriever | format_docs,          # 检索出文档 -> 拼成字符串
        "input": RunnablePassthrough()               # 把用户原始问题原封不动传过来
    }
    # 第二步：自动填坑（相当于自动调用 prompt.format()）
    | prompt
    # 第三步：把填好的提示词丢给你的本地 Qwen 模型
    | qwen3_06b_llm
    # 第四步：解析模型输出，变成干净的字符串
    | StrOutputParser()
)
print("✅ RAG 链构建完成！")

✅ RAG 链构建完成！


**使用语言模型来测试**

In [21]:
query = "Agent中的RAG是什么？"

# 运行RAG链
response = rag_chain.invoke(query)
print("🤖 Qwen3 回答：")
print(response)


🧠 [模型思考]: <think>
好的，我现在要回答用户的问题：“Agent中的RAG是什么？”根据提供的已知信息，我需要仔细分析和整理相关内容。

首先，用户的问题是关于Agent中的RAG（检索-生成）机制。根据已知信息，RAG的核⼼原理是结合“检索”和“生成”，当用户提出查询时，系统首先通过检索模块找到相关文本段落，然后将这些段落作为附加信息传递给语言模型，模型据此生成更准确的回答。同时，RAG能够缓解大模型的“幻觉”问题，因为生成的内容基于真实文档，具有可追溯性和可信度。此外，RAG还加快了知识更新速度，及时反映最新的领域动态。

接下来，我需要将这些信息组织成一个清晰的回答。需要注意的是，用户可能希望得到简明扼要的解释，同时涵盖关键点，如检索和生成的作用、缓解幻觉、知识更新等。同时，要确保回答符合问题的要求，即明确说明RAG在Agent中的具体作用。

在回答时，应该分点说明，但根据用户的问题，可能只需要简要总结。例如，可以指出RAG是Agent的一部分，结合检索和生成，确保回答准确且符合已知信息。
</think>

🤖 Qwen3 回答：
Agent中的RAG（检索-生成）是指将检索和生成结合的机制，具体作用如下：

1. **检索**：系统通过编码器将文档库分割为短片段，并构建向量索引，以便快速定位与问题相关的内容。  
2. **生成**：将检索到的上下文信息作为输入，生成精准且可信的回答，确保内容基于真实文档。  
3. **作用**：缓解大模型的“幻觉”问题，提升回答的可追溯性和可信度；同时加快知识更新速度，及时反映领域动态。  

RAG通过将检索与生成整合，增强了模型对真实信息的依赖，从而优化了回答的准确性与时效性。


看一下上述回答的参考资料

In [23]:
# 也可以顺便看看检索到的参考资料
docs = retriever.invoke(query)
print("\n📚 参考来源：")
for i, doc in enumerate(docs, 1):
    print(f"--- 来源 {i} (第 {doc.metadata.get('page', '?')+1} 页) ---")
    print(doc.page_content[:200] + "...\n")


📚 参考来源：
--- 来源 1 (第 153 页) ---
内容更加符合实时性要求。
RAG 的核⼼原理在于将“检索”与“⽣成”结合：当⽤户提出查询时，系统⾸先通过检索模块找到与问题相关的⽂本⽚
段，然后将这些⽚段作为附加信息传递给语⾔模型，模型据此⽣成更为精准和可靠的回答。通过这种⽅式，RAG 有
效缓解了⼤语⾔模型的“幻觉”问题，因为⽣成的内容建⽴在真实⽂档的基础上，使得答案更具可追溯性和可信度。
同时，由于引⼊了最新的信息源，RAG 技术⼤⼤加快了知...

--- 来源 2 (第 162 页) ---
注：7.2 章节的所有代码均可在 Happy-LLM Chapter7 RAG 中找到。
7.3 Agent  
7.3.1 什么是 LLM Agent？  
简单来说，⼤模型Agent是⼀个以LLM为核⼼“⼤脑”，并赋予其⾃主规划、记忆和使⽤⼯具能⼒的系统。 它不再仅
仅是被动地响应⽤户的提示（Prompt），⽽是能够：
1. 理解⽬标（Goal Understanding）： 接收⼀个相对复杂...

--- 来源 3 (第 154 页) ---
图7.5 TinyRAG 项⽬结构
接下来，让我们梳理⼀下RAG的流程是什么样的呢？
索引：将⽂档库分割成较短的⽚段，并通过编码器构建向量索引。
检索：根据问题和⽚段的相似度检索相关⽂档⽚段。
⽣成：以检索到的上下⽂为条件，⽣成问题的回答。
如下图7.6所示的流程图，图⽚出处 Retrieval-Augmented Generation for Large Language Models: A S...



In [24]:
del model
del tokenizer
CleanMemory()

#### 其他模型写法

**如果使用OpenAI API 模型，可以这样写**
```python
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="xxxx-xx",                  # 换成你想用的模型，比如 gpt-3.5-turbo, gpt-4o, deepseek-chat 等
    base_url="https://xx.xxx.xx",     # 换成你的 API 提供商的 base_url
    api_key="xx-xxxxxxxxxxxxxxxx",    # 换成你的真实 API Key, 本地部署的可以填任意字符串
    temperature=0.1, # 让回答更基于事实
)

# rag_chain = (
#     {
#         "context": retriever | format_docs,   
#         "input": RunnablePassthrough()          
#     }
#     | prompt
#     | llm      <---------------------只改这里
#     | StrOutputParser()
# )
```

**还可以将我们的模型升级成BaseChatModel，无需部署**


In [34]:
from langchain_core.language_models.chat_models import BaseChatModel, ChatResult, ChatGeneration
from langchain_core.messages import AIMessage, BaseMessage
from typing import Optional, List, Any

class QwenChatModel(BaseChatModel):
    model: Any
    tokenizer: Any

    @property
    def _llm_type(self) -> str:
        return "local-qwen3-chat"

    # 🌟 核心修改：接收的是结构化的 messages 列表
    def _generate(self, messages: List[BaseMessage], stop: Optional[List[str]] = None, **kwargs) -> Any:
        
        # 1. 把 LangChain 的 Message 格式转换成我们函数需要的 字典 格式
        my_messages = []
        for msg in messages:
            if msg.type == "system":
                my_messages.append({"role": "system", "content": msg.content})
            elif msg.type == "human":
                my_messages.append({"role": "user", "content": msg.content})
            elif msg.type == "ai":
                my_messages.append({"role": "assistant", "content": msg.content})
        
        # 2. 调用我们的推理函数
        think_output, output = get_qwen_output(my_messages, self.model, self.tokenizer)
        
        # 3. 包装
        # AIMessage
        ai_message = AIMessage(content=output)
        # ChatGeneration
        generation = ChatGeneration(message=ai_message)
        # ChatResult
        return ChatResult(generations=[generation])

In [35]:
model, tokenizer = load_model()
llm = QwenChatModel(model=model, tokenizer=tokenizer)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In [38]:
template = """已知信息：
{context}

请根据上面的已知信息回答问题。
问题：{input}
回答："""
prompt = PromptTemplate.from_template(template)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
rag_chain = (
    {
        "context": retriever | format_docs,      
        "input": RunnablePassthrough()            
    }
    | prompt
    | llm
    | StrOutputParser()
)
query = "Agent中的RAG是什么？"

# 运行RAG链
response = rag_chain.invoke(query)
print("🤖 Qwen3 回答：")
print(response)

🤖 Qwen3 回答：
Agent中的RAG（Retrieval-Augmented Generation）是指Agent通过检索相关文档或信息，以增强生成内容的准确性和可靠性。具体来说，RAG通过结合检索模块找到与问题相关的文本段落，并将这些段落作为附加信息传递给语言模型，从而生成更精准、可信的回答。这有效缓解了大语言模型的“幻觉”问题，确保回答基于真实数据，同时加快知识更新速度。


In [42]:
del model
del tokenizer
del llm
CleanMemory()

## Agentic RAG 

Agentic RAG = 把 RAG 变成 Agent 的一个“工具”，由智能体自主决定：要不要检索、查几次、查哪些库、用不用其他工具、查完还要不要再查

### 简单代码模拟

In [15]:
import chromadb

# 1. 初始化 ChromaDB 客户端（内存模式，不用装数据库）
client = chromadb.Client()

# 2. 创建一个集合
collection = client.get_or_create_collection(name="financial_report")

# 3. 插入模拟文档（故意用“营收”而不是“收入”）
docs = [
    "第一季度(Q1)销售额为120万，主要靠新产品上线拉动。",
    "第二季度(Q2)营收达到了150万，市场推广效果显著。", # 注意这里是“营收”
    "第三季度(Q3)销售额小幅回落至130万，受季节性影响。"
]
collection.add(
    documents=docs,
    ids=["q1", "q2", "q3"]
)
print("✅ 数据初始化完成！")

/home/kokomi/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|█| 79.3M/79.3M [01:37<00:00, 84


✅ 数据初始化完成！


传统RAG

In [16]:
def traditional_rag(query):
    print(f"\n🔍 [传统RAG] 收到问题: {query}")
    
    # 1. 直接用原问题检索（只搜一次）
    results = collection.query(query_texts=[query], n_results=1)
    context = results['documents'][0][0]
    print(f"📄 [传统RAG] 检索到的上下文: {context}")
    
    # 2. 拼装 Prompt 让大模型生成（这里用 print 模拟大模型生成）
    prompt = f"根据以下资料回答问题：\n资料：{context}\n问题：{query}"
    
    # 模拟输出：因为搜的是"收入"，而文档里是"营收"，可能搜不到 Q2 的数据，只能搜到 Q1 或 Q3
    print(f"🤖 [传统RAG] 最终生成的 Prompt: {prompt}")
    return "（基于可能残缺的上下文生成的答案）"

# 测试：用户问“收入”，但文档里是“营收”，向量检索可能匹配不到最准的 Q2 数据
traditional_rag("公司Q2的收入是多少？")


🔍 [传统RAG] 收到问题: 公司Q2的收入是多少？
📄 [传统RAG] 检索到的上下文: 第一季度(Q1)销售额为120万，主要靠新产品上线拉动。
🤖 [传统RAG] 最终生成的 Prompt: 根据以下资料回答问题：
资料：第一季度(Q1)销售额为120万，主要靠新产品上线拉动。
问题：公司Q2的收入是多少？


'（基于可能残缺的上下文生成的答案）'

基于智能体的RAG

In [21]:
def agentic_rag(query):
    print(f"\n🤖 [Agent] 收到任务: {query}")
    
    memory = [] # Agent 的记忆
    max_steps = 3 # 硬性兜底：最多重试 3 次（Loop 工程的熔断机制）
    
    for step in range(max_steps):
        print(f"\n--- 🔄 Loop 第 {step + 1} 轮 ---")
        
        # 👉 步骤 A：规划（模拟 LLM 思考决定用什么工具和参数）
        if step == 0:
            thought = "我需要先查一下 Q2 的收入数据。"
            action = "search_db"
            action_input = "Q2 收入" # 第一次搜，用了和用户一样的词
        elif step == 1:
            thought = "刚才没搜到 Q2 数据，可能是用词不对，我换词重搜。"
            action = "search_db"
            action_input = "第二季度 Q2 收入" # 反思后，改写查询词（Query Rewriting）
        else:
            thought = "我已经找到了 Q2 营收是 150 万，可以回答了。"
            action = "finish"
            action_input = "公司 Q2 的收入（营收）为 150 万。"

        print(f"🧠 [思考] {thought}")
        print(f"🛠️ [动作] {action}({action_input})")
        
        # 👉 步骤 B：执行动作（调用工具）
        if action == "search_db":
            results = collection.query(query_texts=[action_input], n_results=1)
            observation = results['documents'][0][0]
            print(f"👀 [观察] 搜到了: {observation}")
            
            # 👉 步骤 C：评估（模拟 LLM 判断结果够不够）
            if "Q2" in observation and ("营收" in observation or "收入" in observation):
                print("✅ [评估] 信息已经足够！")
                memory.append(f"已找到关键数据：{observation}")
            else:
                print("⚠️ [评估] 搜到的不是 Q2 数据，信息不足，需要重试。")
                memory.append("第一次搜索失败，没找到 Q2数据。")
                
        elif action == "finish":
            print(f"🏁 [结束] 最终答案: {action_input}")
            return action_input

    print("🛑 [系统熔断] 达到最大步数，强制停止！")
    return "抱歉，我未能找到答案。"

# 测试同样的疑问
agentic_rag("公司Q2的收入是多少？")


🤖 [Agent] 收到任务: 公司Q2的收入是多少？

--- 🔄 Loop 第 1 轮 ---
🧠 [思考] 我需要先查一下 Q2 的收入数据。
🛠️ [动作] search_db(Q2 收入)
👀 [观察] 搜到了: 第一季度(Q1)销售额为120万，主要靠新产品上线拉动。
⚠️ [评估] 搜到的不是 Q2 数据，信息不足，需要重试。

--- 🔄 Loop 第 2 轮 ---
🧠 [思考] 刚才没搜到 Q2 数据，可能是用词不对，我换词重搜。
🛠️ [动作] search_db(第二季度 Q2 收入)
👀 [观察] 搜到了: 第二季度(Q2)营收达到了150万，市场推广效果显著。
✅ [评估] 信息已经足够！

--- 🔄 Loop 第 3 轮 ---
🧠 [思考] 我已经找到了 Q2 营收是 150 万，可以回答了。
🛠️ [动作] finish(公司 Q2 的收入（营收）为 150 万。)
🏁 [结束] 最终答案: 公司 Q2 的收入（营收）为 150 万。


'公司 Q2 的收入（营收）为 150 万。'

##  MCP

MCP（Model Context Protocol）：模型上下文协议，统一、结构化地给模型提供工具描述 & 数据上下文

MCP 是 C/S 架构的。

Agent 代码是 MCP Client（客户端），而提供工具的程序是 MCP Server（服务端）。

### 简单示例演示

In [5]:
# !pip install mcp
!pip show mcp

Name: mcp
Version: 1.27.1
Summary: Model Context Protocol SDK
Home-page: https://modelcontextprotocol.io
Author: Anthropic, PBC.
Author-email: 
License: MIT
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: anyio, httpx, httpx-sse, jsonschema, pydantic, pydantic-settings, pyjwt, python-multipart, sse-starlette, starlette, typing-extensions, typing-inspection, uvicorn
Required-by: 


模拟 MCP 服务端

In [16]:
from mcp.server.fastmcp import FastMCP

# 1. 初始化 MCP Server
mcp = FastMCP("SimpleServer")

# 2. 定义工具
@mcp.tool()
def add2nums(a: int, b: int) -> int:
    """将两个数字相加并返回结果"""  # ⚠️ 这句话非常重要！MCP 会把它作为工具描述发给 LLM
    return a + b

# @mcp.tool()
# def sub2nums(a: int, b: int) -> int:
#     """将两个数字相减并返回结果""" 
#     return a - b

# @mcp.tool()
# def mult2nums(a: int, b: int) -> int:
#     """将两个数字相乘并返回结果""" 
#     return a * b

获取 MCP 自动生成的工具列表

In [20]:
# 必须加 await
tools_list = await mcp.list_tools()

Agent要用MCP，会先发送一个请求获取工具列表  
打印给 MCP 传给 客户端LLM 看的工具描述 (JSON Schema)

In [18]:
import json
print(json.dumps([tool.model_dump() for tool in tools_list], indent=2, ensure_ascii=False))

[
  {
    "name": "add2nums",
    "title": null,
    "description": "将两个数字相加并返回结果",
    "inputSchema": {
      "properties": {
        "a": {
          "title": "A",
          "type": "integer"
        },
        "b": {
          "title": "B",
          "type": "integer"
        }
      },
      "required": [
        "a",
        "b"
      ],
      "title": "add2numsArguments",
      "type": "object"
    },
    "outputSchema": {
      "properties": {
        "result": {
          "title": "Result",
          "type": "integer"
        }
      },
      "required": [
        "result"
      ],
      "title": "add2numsOutput",
      "type": "object"
    },
    "icons": null,
    "annotations": null,
    "meta": null,
    "execution": null
  }
]


模拟客户端进行工具调用  
假设Agent选择了 add2nums 这个工具，并附带了参数，将这些信息传给MCP服务端得到返回的结果

In [32]:
tool_name = "add2nums"
tool_args = {"a": 5, "b": 3}

# 必须加 await
result = await mcp.call_tool(tool_name, tool_args)
print(f"{result=}")

# 提取并打印文本结果
print(f"调用工具 '{tool_name}' 的结果: {result[0][0].text}")

result=([TextContent(type='text', text='8', annotations=None, meta=None)], {'result': 8})
调用工具 'add2nums' 的结果: 8


### 实际场景使用的流程图

一般仅需配置好对应MCP的请求链接即可使用

```mermaid
sequenceDiagram
    participant Agent as 🤖 Agent (客户端)
    participant Server as 🛠️ MCP Server

    Note over Agent,Server: 阶段一：发现工具 (tools/list)
    Agent->>Server: JSON-RPC 请求: tools/list
    Server-->>Agent: 返回工具清单<br/>(name, description, inputSchema)

    Note over Agent,Server: 阶段二：调用工具 (tools/call)
    Agent->>Server: JSON-RPC 请求: tools/call<br/>(name: "add", arguments: {a: 1, b: 2})
    Server-->>Agent: 返回执行结果<br/>(content: [{type: "text", text: "3"}])

## Skills

- Tool（工具）：原子操作，比如内置或者mcp提供的 search_web()、read_file()
- Skill（技能）：是 Agent 侧的工作流，编排一个或多个 Tool，描述一个抽象的任务需要如来分步完成

### 示例展示：**当前热搜榜单汇总提取**技能

Skill名称：五大平台热搜前十提取+整合（微博/小红书/抖音/百度/贴吧）
触发指令：当用户输入以下任意指令触发，示例包括：  
- 提取五大平台热搜前十、  
- 查微博小红书抖音百度贴吧各前十热搜、  
- 汇总五大平台热门前十、  
- 获取五大平台热搜前十并整合、  
- 查各平台热搜前十再合并

---

执行规则（严格遵循，兼顾合规、无反爬、步骤明确）  
1. 检索平台及具体网页地址（均为公开无登录、无反爬入口，国内可直接访问）：
              微博：微博热搜榜网页版，具体地址：https://s.weibo.com/top/summary
2. 小红书：小红书公开热门话题板块（发现页），具体地址：https://www.xiaohongshu.com/explore 
3. 抖音：抖音网页版公开热点榜，具体地址：https://www.douyin.com/hot，遇到登录直接关闭登录窗口
4. 百度：百度热搜公开版，具体地址：https://top.baidu.com
5. 贴吧：百度贴吧公开热议榜，具体地址：http://c.tieba.baidu.com/hottopic/browse/topicList?res_type=1 ，无需登录即可访问，遇到登录直接关闭登录窗口
6. 检索范围补充：仅提取上述五个平台公开热门榜单的前十内容，若某平台网页访问失败，不额外操作，直接检索下一个平台。
7. 检索核心与具体步骤（严格按以下步骤执行）：
  - 第一步：检索微博热搜榜网页版，提取实时前十内容，记录每条热搜的「排名、事件名称、热度值（可获取则补充）」；若微博网页访问失败，直接执行第二步；
  - 第二步：检索小红书公开热门话题板块，提取热门前十内容，记录每条热门的「排名、话题/事件名称、参与人数/点赞量（可获取则补充）」；若小红书网页访问失败，直接执行第三步；
  - 第三步：检索抖音网页版公开热点榜，提取热门前十内容，记录每条热门的「排名、话题/事件名称、热度值（可获取则补充）」；若抖音网页访问失败，直接执行第四步；
  - 第四步：检索百度热搜公开版，提取实时前十内容，记录每条热搜的「排名、事件名称、热度值（可获取则补充）」；若百度网页访问失败，直接执行第五步；
  - 第五步：检索百度贴吧公开热议榜，提取热门前十内容，记录每条热门的「排名、话题/事件名称、参与人数/互动量（可获取则补充）」；若贴吧网页访问失败，直接进入整合步骤
  - 第六步：整合汇总，对所有成功检索到的平台内容进行去重（相同事件/话题合并），标注该事件涉及的所有平台及对应排名，最终按「综合热度（各平台热度均值）」排序，形成15-20条的整合榜单，兼顾热度与多样性。

8. 输出要求：结构化呈现，直接呈现整合去重后的榜单，无需分平台单独呈现，语言简练、信息完整，不冗余，控制输出长度，避免无效token消耗，标注检索时间、各事件涉及平台及对应排名、综合热度，同时标注跳过的平台（若有网页访问失败）。
9. 补充说明：【若某平台网页访问失败，不额外提示，直接检索下一个平台；所有平台检索完成后，在输出结果标注跳过的平台（若有）；若多个平台出现相同事件，整合时合并为一条，标注涉及的所有平台及对应排名；热度值优先提取平台官方数据，无法获取则标注“热度未公开”】。

---

固定输出模板（自动按此生成，步骤清晰、贴合需求）  
四大平台热搜整合汇总报告（仅呈现去重后榜单）  
检索时间：2026-03-08 16:45（实时生成，精确到分钟）  
检索范围：微博热搜前十、小红书热门前十、抖音热门前十、百度热搜前十、贴吧热门前十（公开无登录、无反爬平台）  
执行步骤：1. 按顺序检索五大平台，网页访问失败直接下一个 → 2. 对成功检索内容去重整合 → 3. 按综合热度排序（15-20条）  
跳过平台说明（若有）：无（如：小红书网页访问失败，已跳过；无则填“无”）  
一、四大平台热搜整合汇总（去重后，按综合热度排序，共15-20条）  
说明：合并各平台相同事件/话题，标注涉及平台及对应排名，综合热度=各平台热度均值（无法获取热度则按参与平台数量排序）；仅呈现成功检索到的平台内容。
1. 综合排名：1 | 事件/话题名称：2026马年春晚最新嘉宾阵容 | 涉及平台：微博+百度+抖音
                  各平台排名：微博第1名、百度第2名、抖音第3名
2. 综合排名：2 | 事件/话题名称：打工人必看的高效摸鱼小技巧 | 涉及平台：小红书+抖音
                  各平台排名：小红书第1名、抖音第4名
3. 综合排名：3 | 事件/话题名称：国产新手机发布价格跌破3000 | 涉及平台：微博+百度
                  各平台排名：微博第3名、百度第1名
......  

--- 

补充说明  
- 1.  时效性：所有热搜均为检索时刻实时数据；
- 2.  数据说明：热度值优先提取平台官方数据，微博以纯数字热度值呈现，小红书以参与人数/点赞量呈现，贴吧以参与人数/互动量呈现，无法获取则标注“热度未公开”；
- 3.  访问失败说明：检索时按“微博→小红书→抖音→百度→贴吧”顺序执行，某平台网页访问失败则直接下一个，不进行任何修复操作或强制跳转，仅在结果标注跳过平台；
- 4.  延伸检索：若需查看某一事件的详细内容，可告知具体事件名称，将补充检索该事件的完整信息（仅提取各平台公开内容）；
- 5.  合规说明：所有内容均来自各平台公开榜单，剔除低俗、敏感内容，严格遵循各平台robots协议，不绕过反爬机制，确保检索行为合规。

### Skills调用流程

```mermaid
flowchart TD
    User([🧑‍💻 用户输入任务]) --> LLM[🧠 LLM 决策]
    
    subgraph Agent [🤖 Agent 客户端]
        Skills -->|提取所需工具| PromptBuilder[组装 Prompt]
        
        GetInternal[获取内置工具描述] --> PromptBuilder
        GetMCP[获取 MCP tools/list] --> PromptBuilder
        
        PromptBuilder --> LLM[🧠 LLM 决策]
        LLM[🧠 LLM 决策] --> tool[🛠️工具]
        tool[🛠️工具] --返回结果--> LLM[🧠 LLM 决策]
        LLM[🧠 LLM 决策] --> Skills
        LLM[🧠 LLM 决策] --> GetInternal
        LLM[🧠 LLM 决策] --> GetMCP
        
    end
    
    LLM -->|任务完成| FinalAnswer([📄 最终回答])

# Harness 工程（2026年2月最新）

2026 年 2 月 HashiCorp / OpenAI 提出 Agent Harness（智能体驾驭工程）

**核心定义**：包裹在 LLM 外围的确定性代码和系统架构，负责调度流转、安全保障、环境交互和结果验证。它解决的是 “Agent 怎么做” 的问题。LLM 是非确定性的，Harness 必须是确定性的。

## 工具执行沙箱

**OS级沙箱**
- 代表工具：KVM、QEMU、VirtualBox、VMware、WSL2
- 特点
  - ✅ 隔离等级最高，几乎无法逃逸，适合运行恶意程序、高风险代码
  - ❌ 资源开销大、启动慢、占用磁盘 / 内存多
- 适用场景：病毒分析、高危程序运行、多系统环境测试

**容器级沙箱**
- 系统容器（完整Linux环境）：LXC / LXD，接近独立系统，内部可启停服务、装软件，比虚拟机轻很多
- 应用容器（单一应用隔离）：Docker、Podman，轻量、启动快、生态极强，云服务、在线代码执行平台标配；但是共享内核，安全弱于虚拟机，适合可信 / 低风险业务

**内核原生进程级沙箱**
- 核心：只隔离单个 / 一组进程，不构建完整运行环境，是容器、firejail等的底层基础。
- 代表工具：unshare、原生 seccomp、chroot、ulimit
- 特点：零额外依赖、性能极致，灵活组合使用
- 适用场景：自定义沙箱、脚本 / 命令临时隔离、嵌入到程序内部做防护
- 核心技术：
  - Namespace：隔离 PID、网络、挂载、用户等资源（unshare 命令）
  - Seccomp：系统调用防火墙，拦截高危内核调用
  - chroot：切换根目录，限制文件访问
  - Cgroups：限制 CPU、内存、磁盘 IO
  - Capabilities：剥离系统特权

**应用/语言级沙箱**
- 命令黑名单沙箱：直接代码拦截 curl、sudo、rm -rf 等高危字符串
- 编程语言专属沙箱：禁用危险模块（如 Python os/subprocess/socket、Node.js 危险 API）
- 特点：实现成本最低，无系统兼容问题，一般作为内核沙箱的补充防护

常见agent的沙箱设计权重图（豆包统计的）

```mermaid
sankey

Container Sandbox,Codex,4
Kernel Process Sandbox,Codex,3
App and Lang Sandbox,Codex,2

OS Level Sandbox,Claude Code,2
Container Sandbox,Claude Code,4
Kernel Process Sandbox,Claude Code,3
App and Lang Sandbox,Claude Code,2

Container Sandbox,OpenClaw,3
Kernel Process Sandbox,OpenClaw,3
App and Lang Sandbox,OpenClaw,1

Container Sandbox,Hermes,3
Kernel Process Sandbox,Hermes,3
App and Lang Sandbox,Hermes,2

OS Level Sandbox,Zhipu Web Agent,2
Container Sandbox,Zhipu Web Agent,4
Kernel Process Sandbox,Zhipu Web Agent,3
App and Lang Sandbox,Zhipu Web Agent,3

OS Level Sandbox,MiniMax,2
Container Sandbox,MiniMax,4
Kernel Process Sandbox,MiniMax,3
App and Lang Sandbox,MiniMax,3

Container Sandbox,copaw,3
Kernel Process Sandbox,copaw,3
App and Lang Sandbox,copaw,1

Container Sandbox,Trae,4
Kernel Process Sandbox,Trae,3
App and Lang Sandbox,Trae,2

### 沙箱体验（使用容器级轻量化方案+黑名单命令拦截方案）

先安装 firejail ，是业界常用的简易 OS 沙盒，在命令行里执行
```bash
sudo apt install firejail -y

使用 container=lxc 骗过firejail，否则在 wsl环境下会失败  (解决方法)[https://github.com/netblue30/firejail/discussions/6177]

In [4]:
!container=lxc firejail --noprofile bash -c "echo 正常运行; pwd"

Parent pid 5821, child pid 5822
]0;firejail bash -c echo 正常运行; pwd Child process initialized in 18.05 ms
正常运行
/home/kokomi/workspace/Artificial-Intelligence-Learning/kkmlmf

Parent is shutting down, bye...


In [3]:
import subprocess

def sandbox_run(cmd: str):
    # 前置补充：弥补WSL2网络隔离失效，做命令黑名单拦截
    ban_list = {"curl", "wget", "ping", "nc", "sudo", "rm -rf", "chmod"}
    cmd_lower = cmd.lower()
    for word in ban_list:
        if word in cmd_lower:
            return f"[沙盒拦截] 禁止指令：{word}"

    # 拼接环境变量 + firejail 完整参数
    args = [
        "env", "container=lxc",
        "firejail",
        "--noprofile",
        "--seccomp",
        "--caps.drop=all",
        "--read-only=/",
        "--read-write=/home/kokomi",
        "--private-tmp",
        "bash", "-c", cmd
    ]
    proc = subprocess.Popen(
        args,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    out, err = proc.communicate()
    return f"OUT:\n{out}\nother:\n{err}"

# 测试用例
print("=== 正常命令 ===")
print(sandbox_run("echo hello && pwd"))

print("=== 系统目录写入(seccomp/只读拦截) ===")
print(sandbox_run("mkdir /test"))

print("=== 网络命令(上层黑名单拦截) ===")
print(sandbox_run("curl baidu.com"))

=== 正常命令 ===
OUT:
hello
/home/kokomi/workspace/Artificial-Intelligence-Learning/kkmlmf

other:
Parent pid 5805, child pid 5806
Child process initialized in 16.45 ms

Parent is shutting down, bye...

=== 系统目录写入(seccomp/只读拦截) ===
OUT:

other:
Parent pid 5809, child pid 5810
Child process initialized in 17.90 ms
mkdir: cannot create directory ‘/test’: Read-only file system

Parent is shutting down, bye...

=== 网络命令(上层黑名单拦截) ===
[沙盒拦截] 禁止指令：curl


## 路由

定义：在多智能体 / 多模型系统里，把任务、请求、子问题分发到最合适的 Agent / 模型 / 工具 的调度层，属于 Harness 的 “交通指挥”。

为不同的任务分配不同的资源，就像公司前台，如果客户来问技术问题，就转接技术部；来问退货，就转接客服部。  
例子：简单问答→小模型；复杂代码→大模型；查数据→专用工具 Agent

### 简单基于字符串匹配的代码模拟（实际可能使用LLM来分配）

In [2]:
class SimpleRouter:
    def __init__(self):
        # 注册路由规则：关键词 -> 对应的处理函数
        self.routes = {
            "天气": self.handle_weather,
            "搜索": self.handle_search,
        }
    
    def add_route(self, keyword, handler):
        self.routes[keyword] = handler

    def route(self, user_input: str):
        print(f"🚦 路由正在分析: {user_input}")
        
        # 遍历关键词，看用户输入里包含什么
        for keyword, handler in self.routes.items():
            if keyword in user_input:
                print(f"➡️ 匹配到关键词 '{keyword}'，转发给对应处理器")
                return handler(user_input)
        
        print("➡️ 未匹配到特定路由，转交大模型自由回答")
        return self.default_handler(user_input)

    # --- 以下是具体的处理函数模拟 ---
    def handle_weather(self, text):
        return "【请求get_weather工具调用】"

    def handle_search(self, text):
        return "【请求web_search工具调用】"

    def default_handler(self, text):
        return "你好呀，我是AI助手！"

# 测试路由
router = SimpleRouter()
print(router.route("帮我查一下苏州的天气"))
print("-" * 30)
print(router.route("搜索与LLM有关的学习资料"))
print("-" * 30)
print(router.route("你好"))

🚦 路由正在分析: 帮我查一下苏州的天气
➡️ 匹配到关键词 '天气'，转发给对应处理器
【请求get_weather工具调用】
------------------------------
🚦 路由正在分析: 搜索与LLM有关的学习资料
➡️ 匹配到关键词 '搜索'，转发给对应处理器
【请求web_search工具调用】
------------------------------
🚦 路由正在分析: 你好
➡️ 未匹配到特定路由，转交大模型自由回答
你好呀，我是AI助手！


## 可观测性

定义：对 Agent 全链路（决策、工具调用、记忆、输出、错误）做日志、追踪、指标、审计，让 Agent 从黑盒变透明，是 Harness 的 “监控与诊断系统”。

核心三要素
- 执行追踪（Traces）：每一步思考、工具调用、输入输出全链路记录
- 日志（Logs）：错误、异常、拒绝动作、关键决策点
- 指标（Metrics）：成功率、错误率、耗时、token 消耗、成本

目的：出问题能定位、能复盘、能归因、能持续改进 Harness 本身

In [8]:
import time
from datetime import datetime

def now(): return datetime.now().strftime("%H:%M:%S")

RED = "\033[91m"    # 开始红色
GREEN = "\033[92m"  # 开始绿色
RESET = "\033[0m"   # 恢复默认颜色

traces = []  # 📍 执行追踪：记录每一步的输入输出
logs = []    # 📝 日志：记录关键事件和报错
metrics = {"成功次数": 0, "失败次数": 0, "总耗时": 0.0}  # 📈 仪表盘：记录统计数字

def agent_run(city):
    print(f"\n--- 运行：查询{city}天气 ---")
    traces.append(f"{now()} [INFO] [思考] 我要查{city}的天气")
    
    # 模拟工具调用
    traces.append(f"{now()} [INFO] [行动] 调用 get_weather(城市={city})")
    start = time.time()
    time.sleep(0.5)                      # 模拟网络延迟
    
    if city == "火星":
        logs.append("❌ ERROR: 查火星天气失败，工具不支持！")
        traces.append(f"{now()} {RED}[ERR] [观察] 调用报错{RESET}")
        metrics["失败次数"] += 1
    else:
        # ✅ 正常情况
        logs.append(f"ℹ️ INFO: 成功获取{city}天气")
        traces.append(f"{now()} [INFO] {GREEN}[观察] 返回结果：晴天32度{RESET}")
        metrics["成功次数"] += 1
    
    metrics["总耗时"] += time.time() - start
    traces.append(f"{now()} [INFO] [输出] 给用户最终回答")

# 跑两次测试
agent_run("苏州")
agent_run("火星")

# 打印结果
print("\n📍 Traces (录像带):", *traces, sep="\n")
print("\n📝 Logs (案卷):", *logs, sep="\n")
print("\n📈 Metrics (仪表盘):", metrics)



--- 运行：查询苏州天气 ---

--- 运行：查询火星天气 ---

📍 Traces (录像带):
10:58:53 [INFO] [思考] 我要查苏州的天气
10:58:53 [INFO] [行动] 调用 get_weather(城市=苏州)
10:58:54 [INFO] [观察] 返回结果：晴天32度
10:58:54 [INFO] [输出] 给用户最终回答
10:58:54 [INFO] [思考] 我要查火星的天气
10:58:54 [INFO] [行动] 调用 get_weather(城市=火星)
10:58:54 [ERR] [观察] 调用报错
10:58:54 [INFO] [输出] 给用户最终回答

📝 Logs (案卷):
ℹ️ INFO: 成功获取苏州天气
❌ ERROR: 查火星天气失败，工具不支持！

📈 Metrics (仪表盘): {'成功次数': 1, '失败次数': 1, '总耗时': 1.000352382659912}


## 评估

定义：在 Agent 执行过程 / 结束后，用自动化规则 + 指标判断输出质量、正确性、安全性、合规性，并决定是否通过、重试、回滚或人工介入，是 Harness 的 “质量关卡”。

核心评估维
- 结果评估：答案是否正确、是否符合要求
- 过程评估：步骤是否合理、有没有幻觉、有没有越权
- 安全评估：是否危险操作、是否泄露信息
- 自动闭环：不达标→重试 / 修正 / 回滚；达标→继续 / 放行

目的：保证 Agent 输出可靠、符合标准，防止错误滚雪球

In [12]:
def out(question):
    # （实际代码中，这里会调用你的LLM和工具）
    if "天气" in question and "苏州" in question:
        return "苏州今天天气是晴天，32度"
    if "下雨" in question and "北京" in question:
        return "北京今天没下雨，是阴天"  # 回答里没有"雨"字，这题会扣分！
    return "我不知道"

test_cases = [
    {"question": "苏州天气怎样？", "expected_keywords": ["苏州", "天气"]},
    {"question": "北京明天会下雨吗？", "expected_keywords": ["北京", "明天", "下雨"]}
]

RED = "\033[91m"    # 开始红色
GREEN = "\033[92m"  # 开始绿色
RESET = "\033[0m"   # 恢复默认颜色
score = 0
for i, case in enumerate(test_cases, 1):
    q = case["question"]
    ks = case["expected_keywords"]
    answer = out(q)                       # Agent回答
    # 看答案里有没有包含所有关键词
    is_pass = all(k in answer for k in ks)
    
    if is_pass:
        score += 1
        print(f"{GREEN}✅ 第{i}题及格 | 问题：{q} | 回答：{answer}{RESET}")
    else:
        missing = [k for k in ks if k not in answer]
        print(f"{RED}❌ 第{i}题不及格 | 问题：{q} | 回答：{answer} | 缺少关键词：{missing}{RESET}")

print(f"\n🎓 最终得分：{score}/{len(test_cases)}")

✅ 第1题及格 | 问题：苏州天气怎样？ | 回答：苏州今天天气是晴天，32度
❌ 第2题不及格 | 问题：北京明天会下雨吗？ | 回答：北京今天没下雨，是阴天 | 缺少关键词：['明天']

🎓 最终得分：1/2


# Loop 工程（2026年6月最新）

**核心定义**：设计“谁在什么时候、用什么 Prompt、带什么 Context、在什么 Harness 里反复跑”的系统。人不再亲自一轮一轮 Prompt，而是设计一个会自己 Prompt Agent 的循环系统。

```mermaid
flowchart LR
  P[Prompt 工程<br/>让模型知道要输出什么<br/>怎么输出] --> C[Context 工程<br/>给模型提供参考资料]
  C --> H[Harness 工程<br/>保证模型的安全性]
  H --> L[Loop 工程<br/>让模型自驱动运行]

  P:::soft
  C:::soft
  H:::soft
  L:::highlight

  classDef soft fill:#f5f5f5,stroke:#555,stroke-width:1px;
  classDef highlight fill:#ffe6b0,stroke:#e6a700,stroke-width:2px;
```

```mermaid
sequenceDiagram
    actor User as 👤 用户
    participant AgentLoop as 🔄 Agent Loop<br/>调度器
    participant LLM as 🧠 LLM<br/>推理
    participant Tools as 🔧 MCP / Skills / Tools<br/>能力接口
    participant Obs as 📊 Observability<br/>可观测性

    User->>AgentLoop: 🚀 下达任务：查苏州天气并推荐景点
    AgentLoop->>Obs: 📍 start_trace()
    AgentLoop->>AgentLoop: 📦 初始化上下文(目标 + 可用Skills列表)

    rect rgb(240, 245, 255)
        Note over AgentLoop,Obs: 第 1 轮循环
        AgentLoop->>LLM: 📤 发送上下文(用户问题 + Skills列表)
        LLM-->>AgentLoop: 📥 Thought: 先查天气<br/>Action: call_skill("weather_query", "苏州")
        AgentLoop->>Obs: 📍📝 记录 Thought + Action
        AgentLoop->>Tools: 🔧 调用 weather_query("苏州")
        Tools-->>AgentLoop: 📥 Observation: 苏州晴天32度
        AgentLoop->>Obs: 📍📝📈 记录 Observation + 耗时
        AgentLoop->>AgentLoop: 📝 评估：查到了天气，目标未完成，继续
        AgentLoop->>AgentLoop: 📝 把 Observation 追加到上下文
    end

    rect rgb(245, 255, 240)
        Note over AgentLoop,Obs: 第 2 轮循环
        AgentLoop->>LLM: 📤 发送更新后的上下文(含天气结果)
        LLM-->>AgentLoop: 📥 Thought: 晴天适合户外，搜景点<br/>Action: call_skill("search", "苏州户外景点")
        AgentLoop->>Obs: 📍📝 记录 Thought + Action
        AgentLoop->>Tools: 🔧 调用 search("苏州户外景点")
        Tools-->>AgentLoop: 📥 Observation: 拙政园和虎丘
        AgentLoop->>Obs: 📍📝📈 记录 Observation + 耗时
        AgentLoop->>AgentLoop: 📝 评估：信息已齐全，准备输出
        AgentLoop->>AgentLoop: 📝 把 Observation 追加到上下文
    end

    rect rgb(255, 250, 240)
        Note over AgentLoop,Obs: 第 3 轮循环 (Final)
        AgentLoop->>LLM: 📤 发送完整上下文
        LLM-->>AgentLoop: 📥 Thought: 任务完成<br/>Finish: 苏州晴天32度，推荐拙政园
        AgentLoop->>Obs: 📍📝 记录最终输出
    end

    AgentLoop-->>User: 💬 苏州晴天32度，推荐去拙政园
    AgentLoop->>Obs: 📍 end_trace() + 📈 汇总 Metrics
    Obs-->>User: 📊 Traces / Logs / Metrics 报告

# Agent开发框架

## 智能体框架

### 单智能体框架

**LangGraph**：
- 作为 LangChain 生态的扩展，LangGraph 另辟蹊径，把 Agent 的"思考-行动-观察"循环画成显式状态图(节点:每一步操作 + 边:跳转逻辑 + 检查点)
- 适合:复杂长任务、要可调试、要审计(金融/医疗)
- 不适合:简单对话

```mermaid
stateDiagram-v2
    direction LR
    [*] --> 思考
    思考 --> 工具调用: 决定调工具
    思考 --> [*]: 直接回答
    工具调用 --> 观察: 工具返回
    观察 --> 思考: 还要继续
    观察 --> [*]: 任务完成
    工具调用 --> 检查点: 状态持久化
    检查点 --> 工具调用
    观察 --> 人机确认: 关键步骤
    人机确认 --> 思考

**AgentScope**：
- 国产通用 Agent 框架,定位和 LangGraph 同级
- 它不仅提供了直观易用的编程接口，更重要的是内置了分布式部署、容错恢复、可观测性等企业级特性，使其特别适合构建需要长期稳定运行的生产环境应用。
- 适合:国内团队、中文场景、要单 Agent 也可能要扩展到多 Agent
- 不适合:只想要最轻量库

```mermaid
graph LR
    subgraph L1[应用层]
        A1[CoPaw 桌面 Agent]
        A2[其他应用]
    end
    subgraph L2[框架层 - AgentScope]
        RA[ReActAgent]
        DA[DialogAgent]
        MA[多 Agent 编排]
    end
    subgraph L3[能力组件]
        T[Tools 工具]
        M[ReMe 记忆]
        MCP[MCP 客户端]
    end
    subgraph L4[模型层]
        Q[Qwen]
        G[GLM]
        L[Llama]
    end
    L1 --> L2
    L2 --> L3
    L2 --> L4

```mermaid
sequenceDiagram
    autonumber
    participant U as 👤 User
    participant H as 📨 MsgHub<br/>(消息总线)
    participant A as 🤖 Agent A<br/>研究员
    participant B as 🤖 Agent B<br/>工程师
    participant C as 🤖 Agent C<br/>审核员
    participant M as 🧠 ReMe<br/>记忆
    participant T as 🔧 MCP<br/>工具

    U->>H: 投递任务:调研+实现+审核
    H->>A: 路由 → Agent A
    A->>M: 读取历史上下文
    M-->>A: 返回相关记忆
    A->>T: 🔧 调搜索/抓取工具
    T-->>A: 返回原始数据
    A->>H: 📤 投递:研究成果
    H->>B: 路由 → Agent B
    B->>M: 💾 写入:方案草稿
    B->>T: 🔧 执行代码/跑测试
    T-->>B: 测试结果
    B->>H: 📤 投递:实现+测试报告
    H->>C: 路由 → Agent C
    C->>M: 读取:待审核产物
    C->>H: ✅ 投递:审核通过
    H->>U: 🎉 最终交付

### 多智能体框架

**AutoGen**：
- AutoGen 的设计哲学根植于"以对话驱动协作"。它巧妙地将复杂的任务解决流程，映射为不同角色的智能体之间的一系列自动化对话。
- 适合:研究、需要人类参与的协作、有多角色对话需求

```mermaid
graph TB
    C((🎯<br/>AutoGen))
    C --> M1[🤝 4 协作模式<br/>GroupChat · Sequential<br/>Hierarchical · Nested]
    C --> M2[🛠️ 7 大能力<br/>人在环·代码沙箱·工具·<br/>RAG·记忆·可观测·分布式]
    C --> M3[🧠 多模型<br/>OpenAI / Anthropic /<br/>Azure / 本地 / 国产]
    C --> M4[🚪 3 入口<br/>SDK · Studio · Bench]

    classDef centerStyle fill:#7c3aed,stroke:#5b21b6,color:#fff,stroke-width:3px
    classDef modeStyle fill:#10b981,stroke:#047857,color:#fff
    classDef capStyle fill:#f59e0b,stroke:#b45309,color:#fff
    classDef modelStyle fill:#3b82f6,stroke:#1d4ed8,color:#fff
    classDef entryStyle fill:#ec4899,stroke:#9d174d,color:#fff

    class C centerStyle
    class M1 modeStyle
    class M2 capStyle
    class M3 modelStyle
    class M4 entryStyle

```mermaid
sequenceDiagram
    participant U as UserProxy
    participant A as Coder
    participant B as Reviewer
    participant C as Tester
    U->>A: 帮我写个排序函数
    A->>B: 提交代码
    B-->>A: 指出 3 个 bug
    A->>A: 修复
    A->>C: 跑测试
    C-->>U: 全过 ✅

**CAMEL**：
- 与 AutoGen 和 AgentScope 这样功能全面的框架不同，CAMEL最初的核心目标是探索如何在最少的人类干预下，让**两个智能体**通过“角色扮演”自主协作解决复杂任务。
- 两大核心概念：角色扮演 (Role-Playing) 和 引导性提示 (Inception Prompting)。
    - 一个扮演“AI 用户” (AI User)，负责提出需求、下达指令和构思任务步骤；另一个则扮演“AI 助理” (AI Assistant)，负责根据指令执行具体操作和提供解决方案。
    - “引导性提示”是在对话开始前，分别注入给两个智能体的一段精心设计的、结构化的初始指令（System Prompt）。包含自身角色、协作者角色、共同目标、行为约束和沟通协议。
- 适合:Agent 协作机制研究、自主智能体行为探索、论文复现
- 不适合:直接做产品(可观测性、稳定性、生态都不如 AutoGen)

```mermaid
sequenceDiagram
    participant SYS as 🎯 System Prompt<br/>(Inception 机制)
    participant U as User Agent<br/>(甲方/需求方)
    participant A as Assistant Agent<br/>(乙方/执行方)
    participant T as 🔧 工具/代码

    SYS->>U: 植入角色:你是任务规划者
    SYS->>A: 植入角色:你是执行者
    U->>A: 📋 派子任务 1
    A->>T: 调工具/写代码
    T-->>A: 返回结果
    A-->>U: ✅ 交付
    U->>A: 📋 派子任务 2
    A->>T: 调工具/写代码
    T-->>A: 返回结果
    A-->>U: ✅ 交付
    Note over U,A: 🔁 循环直到共同目标完成
    A-->>U: 🎉 最终交付

### 与之前的工作流（思维范式）和 OpenClaw等 的关系

```mermaid
graph LR
    L3[📦 应用产品层<br/>OpenClaw 、 Hermes Agent 、 CoPaw 、 Claude Code 、 TRAE SOLO]
    L2[🛠 库级框架层<br/>LangChain 、 LangGraph 、 AutoGen 、 CrewAI<br/>AgentScope 、 Smolagents 、 MetaGPT 、 CAMEL]
    L1[💡 思维范式层<br/>ReAct 、 Plan-and-Solve 、 Reflection 、 Tree/Graph of Thoughts]

    L3 -->|基于| L2
    L2 -->|实现| L1

    classDef app fill:#ececff,stroke:#ec174d,color:#000
    classDef lib fill:#7ec6ff,stroke:#1d4ed8,color:#000
    classDef think fill:#fff5ad,stroke:#047857,color:#000
    class L3 app
    class L2 lib
    class L1 think

# 基于LangGraph的Agent 

In [23]:
!pip install langchain langgraph langchain-community langchain-text-splitters chromadb tavily-python requests modelscope

### 导入包并配置模型和apikey

In [1]:
import os
import re
import gc
import time
import subprocess
import torch
import requests
from datetime import datetime
from tavily import TavilyClient
from modelscope import AutoModelForCausalLM, AutoTokenizer
from langgraph.graph import StateGraph, START, END
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import Chroma
from typing import TypedDict, Annotated, Sequence
import operator
import chromadb
from chromadb.utils import embedding_functions

# 配置密钥 & 模型路径（和笔记一致）
os.environ["TAVILY_API_KEY"] = "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"
QWEN_PATH = "./Qwen/Qwen3-0.6B"
EMBED_MODEL_PATH = "./BAAI/bge-small-zh-v1.5"
MEMORY_PERSIST_DIR = "./memory_db"
MEMORY_COLLECTION = "memories"
RAG_DB_DIR = "./rag_chroma_db"

/tmp/ipykernel_8876/295799750.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### 辅助函数

In [2]:
# 1. 内存清理
def CleanMemory():
    import gc
    torch.cuda.empty_cache()
    gc.collect()

# 增强版：解析 Thought + Action
def parse_thought_action(full_text: str):
    thought_reg = r"Thought\s*:\s*(.*?)(?=\s*Action\s*:|$)"
    action_reg = r"Action\s*:\s*(.*?)(?=\n|$)"
    t_match = re.search(thought_reg, full_text, re.DOTALL)
    a_match = re.search(action_reg, full_text, re.DOTALL)
    thought = t_match.group(1).strip() if t_match else ""
    action = a_match.group(1).strip() if a_match else ""
    return thought, action

# 增强版：解析工具调用
def parse_tool_call(action_str: str):
    tool_reg = r"(\w+)\s*\(\s*(.*?)\s*\)"
    kw_reg = r'(\w+)\s*=\s*"([^"]*)"'
    t_match = re.match(tool_reg, action_str)
    if not t_match:
        return None, {}
    tool_name = t_match.group(1)
    args_str = t_match.group(2)
    kwargs = dict(re.findall(kw_reg, args_str))
    return tool_name, kwargs

# 3. 加载本地的小模型
def load_model(model_name="./Qwen/Qwen3-0.6B"):
    # 加载分词器
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # 加载模型
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.bfloat16,
        # dtype=torch.float16, #上面的不能用就用下面的
        device_map="auto"
    )
    return model, tokenizer

# 4.使用模型推理得到输出
def get_qwen_output(messages, model, tokenizer):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False # 是否使用思考模式
    )
    # 编码
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    # 推理
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=32768 #最大上下文长度
    )
    # 得到输出并转 ids
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    try:
        # 找到思考结束的符号的id的位置 151668 (</think>)
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0
    # 思考内容
    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    # 输出内容
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
    print(f"测试输出 ：{content}\n=======")
    return thinking_content, content

def now():
    return datetime.now().strftime("%H:%M:%S")

### 沙箱配置

In [3]:
def sandbox_run(cmd: str):
    ban_list = {"curl", "wget", "ping", "nc", "sudo", "rm -rf", "chmod"}
    cmd_lower = cmd.lower()
    for word in ban_list:
        if word in cmd_lower:
            return f"[沙盒拦截] 禁止指令：{word}"
    # 拼接环境变量 + firejail 完整参数
    args = [
        "env", "container=lxc",
        "firejail",
        "--noprofile",
        "--seccomp",
        "--caps.drop=all",
        "--read-only=/",
        "--read-write=/home/kokomi",
        "--private-tmp",
        "bash", "-c", cmd
    ]
    proc = subprocess.Popen(
        args,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    out, err = proc.communicate()
    return f"OUT:\n{out}"

### RAG

In [4]:
# 1. 初始化嵌入模型
rag_embedding = HuggingFaceBgeEmbeddings(model_name=EMBED_MODEL_PATH)

# 2. 文本切分器
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", "。", "！", "？", " ", ""]
)

# 3. 加载PDF并构建向量库（LLM学习教程）
def init_rag_db(pdf_path: str = "./data/Agent/Happy-LLM-0727.pdf"):
    try:
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
        splits = text_splitter.split_documents(documents)
        vectorstore = Chroma.from_documents(
            documents=splits,
            embedding=rag_embedding,
            # persist_directory=RAG_DB_DIR    # 可以只加载到内存
        )
        vectorstore.persist()
        return vectorstore.as_retriever(search_kwargs={"k": 3})
    except Exception as e:
        print(f"RAG库初始化失败: {e}")
        return None
# 初始化检索器
rag_retriever = init_rag_db()

/tmp/ipykernel_8876/2694936335.py:2: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  rag_embedding = HuggingFaceBgeEmbeddings(model_name=EMBED_MODEL_PATH)


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ./BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_8876/2694936335.py:22: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


### Tools封装

RAG检索

In [5]:
@tool
def rag_search(query: str) -> str:
    """检索知识库内容，用于回答专业问题"""
    if not rag_retriever:
        return "RAG知识库未初始化"
    docs = rag_retriever.invoke(query)
    content = "\n\n".join([doc.page_content for doc in docs])
    return f"RAG检索结果:\n{content}"

沙箱命令执行

In [6]:
@tool
def exec_sandbox_cmd(cmd: str) -> str:
    """在安全沙箱中执行Shell命令，高危指令会被拦截"""
    return sandbox_run(cmd)

查天气

In [7]:
@tool
def get_weather(city: str) -> str:
    """查询指定城市实时天气"""
    url = f"https://wttr.in/{city}?format=j1"
    try:
        response = requests.get(url)
        response.raise_for_status() 
        data = response.json()
        current_condition = data['current_condition'][0]
        weather_desc = current_condition['weatherDesc'][0]['value']
        temp_c = current_condition['temp_C']
        return f"{city}当前天气:{weather_desc}，气温{temp_c}摄氏度"
        
    except requests.exceptions.RequestException as e:
        return f"错误:查询天气时遇到网络问题 - {e}"
    except (KeyError, IndexError) as e:
        return f"错误:解析天气数据失败，可能是城市名称无效 - {e}"

查景点

In [8]:
@tool
def get_attraction(city: str, weather: str) -> str:
    """根据城市和天气推荐旅游景点"""
    print("正在调用tool:get_attraction......")
    api_key = os.environ.get("TAVILY_API_KEY")
    if not api_key:
        return "错误:未配置TAVILY_API_KEY环境变量。"

    tavily = TavilyClient(api_key=api_key)
    query = f"'{city}' 在'{weather}'天气下最值得去的旅游景点推荐及理由"
    
    try:
        response = tavily.search(query=query, search_depth="basic", include_answer=True)
        if response.get("answer"):
            return response["answer"]
        
        formatted_results = []
        for result in response.get("results", []):
            formatted_results.append(f"- {result['title']}: {result['content']}")
        
        if not formatted_results:
             return "抱歉，没有找到相关的旅游景点推荐。"
        return "根据搜索，为您找到以下信息:\n" + "\n".join(formatted_results)

    except Exception as e:
        return f"错误:执行Tavily搜索时出现问题 - {e}"

In [9]:
tools = [get_weather, get_attraction, exec_sandbox_cmd, rag_search]
tool_map = {t.name: t for t in tools}

### 向量记忆系统

初始化

In [10]:
mem_client = chromadb.PersistentClient(path=MEMORY_PERSIST_DIR)
mem_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL_PATH)
mem_collection = mem_client.get_or_create_collection(
    name=MEMORY_COLLECTION,
    embedding_function=mem_ef,
    metadata={"hnsw:space": "cosine"}
)

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ./BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


存储记忆

In [11]:
def remember(text: str, user_id: str = "default", importance: float = 0.5) -> None:
    """存储记忆"""
    mid = f"mem_{int(time.time() * 1_000_000)}"
    mem_collection.add(
        documents=[text],
        metadatas=[{
            "user_id": user_id,
            "importance": importance,
            "created_at": time.time(),
            "last_access": time.time(),
            "access_count": 0
        }],
        ids=[mid]
    )

检索记忆

In [12]:
def recall(query: str, user_id: str = "default", k: int = 3) -> list:
    """检索相关记忆"""
    res = mem_collection.query(
        query_texts=[query],
        n_results=k,
        where={"user_id": user_id}
    )
    return res["documents"][0] if res["documents"] else []

### Skills

创建一个旅行景点查询技能

In [13]:
import os

# 1. 定义路径
SKILL_DIR = "./skills"
SKILL_MD_PATH = os.path.join(SKILL_DIR, "travel_skill.md")

# 2. 技能内容（对应旅行查询技能，纯文本规则，给大模型阅读）
SKILL_MD_CONTENT = """# 旅行景点查询技能
## 技能说明
该技能用于完成城市天气查询+旅游景点推荐的组合任务。
当用户询问某城市天气、旅游攻略、景点推荐相关问题时使用。

## 示例
用户需要查询"南京"的景点推荐，使用get_weather工具，传入参数"南京"，结果发现南京今天下雨；
再使用get_attraction工具，传入参数"南京"、"雨天"，得到结果推荐去xx景点游玩。

## 执行方式
1. 优先使用 get_weather 工具查询目标城市天气
2. 拿到天气结果后，再使用 get_attraction 工具结合目标城市和天气情况推荐景点
3. 整合所有信息，给出最终回答

## 依赖工具
get_weather、get_attraction
"""

if not os.path.exists(SKILL_DIR):
    os.makedirs(SKILL_DIR)
    print(f"✅ 成功创建技能目录: {SKILL_DIR}")

with open(SKILL_MD_PATH, "w", encoding="utf-8") as f:
    f.write(SKILL_MD_CONTENT)
print(f"✅ 技能文件已生成: {SKILL_MD_PATH}")

# 读取MD技能内容函数
def load_skill_content() -> str:
    if not os.path.exists(SKILL_MD_PATH):
        return "暂无可用技能"
    with open(SKILL_MD_PATH, "r", encoding="utf-8") as f:
        return f.read()

# 加载技能文本（后续拼入系统提示词）
skill_text = load_skill_content()
print("\n📄 加载到的技能规则：")
print(skill_text)

✅ 技能文件已生成: ./skills/travel_skill.md

📄 加载到的技能规则：
# 旅行景点查询技能
## 技能说明
该技能用于完成城市天气查询+旅游景点推荐的组合任务。
当用户询问某城市天气、旅游攻略、景点推荐相关问题时使用。

## 示例
用户需要查询"南京"的景点推荐，使用get_weather工具，传入参数"南京"，结果发现南京今天下雨；
再使用get_attraction工具，传入参数"南京"、"雨天"，得到结果推荐去xx景点游玩。

## 执行方式
1. 优先使用 get_weather 工具查询目标城市天气
2. 拿到天气结果后，再使用 get_attraction 工具结合目标城市和天气情况推荐景点
3. 整合所有信息，给出最终回答

## 依赖工具
get_weather、get_attraction



### 系统提示词

In [14]:
SYSTEM_PROMPT = """
你是智能助手。
【可用原子工具】
1. get_weather(city="城市名")：查询指定城市实时天气
2. get_attraction(city="城市", weather="天气")：根据城市+天气推荐景点
3. exec_sandbox_cmd(cmd="命令")：在安全沙箱执行Shell命令
4. rag_search(query="问题")：检索LLM学习笔记知识库内容

【任务还未完成的输出强制格式】
Thought: [你的思考过程]
Action: 工具名(参数="值") 

【任务完成的输出格式】
Thought: [你的思考过程]
Action: Finish[你最终得到的输出内容]

【规则】
1. 请阅读上下文里的技能说明，严格按照技能步骤执行
2. 单次Action只可以用一个工具
3. 在信息收集完毕后并且完成用户的任务后，使用 Finish[你最终得到的输出内容] 输出最终回答
4. 每次只输出两行，一行Thought，一行Action
"""

### Agent全局状态

In [15]:
# ===================== 全局状态 =====================
class AgentState(TypedDict):
    messages: Annotated[Sequence[SystemMessage | HumanMessage | AIMessage], operator.add]
    memory_context: str  # 检索到的历史记忆
    step_count: int       # 循环计数，防止死循环

### 加载 SLM 小语言模型

In [16]:
model, tokenizer = load_model(QWEN_PATH)
print(f"✅ {QWEN_PATH} 模型加载完成")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

✅ ./Qwen/Qwen3-0.6B 模型加载完成


### LangGraph节点和路由

In [17]:
# ===================== 核心节点 =====================
def agent_think_node(state: AgentState):
    """LLM 思考节点：读取技能+记忆，自主规划动作"""
    print("进入思考节点")
    mem_content = state["memory_context"]
    mem_human = HumanMessage(content=f"历史记忆：\n{mem_content}" if mem_content else "无历史记忆")

    recent_messages = state["messages"][-4:]
    
    langchain_messages = [SystemMessage(content=SYSTEM_PROMPT), mem_human] + recent_messages

    messages = []
    for msg in langchain_messages:
        if isinstance(msg, SystemMessage):
            messages.append({"role": "system", "content": msg.content})
        elif isinstance(msg, HumanMessage):
            messages.append({"role": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            messages.append({"role": "assistant", "content": msg.content})
    
    print(f"推理提示词：{messages}")
    think, output = get_qwen_output(messages, model, tokenizer)
    return {
        "messages": [AIMessage(content=output)],
        "step_count": state["step_count"] + 1,
        "memory_context": ""
    }

def memory_node(state: AgentState):
    """记忆检索节点"""
    print(f"进入记忆检索节点")
    if not state["messages"]:
        return {"memory_context": ""}
    user_query = state["messages"][0].content
    mem_list = recall(user_query)
    return {"memory_context": "\n".join(mem_list) if mem_list else ""}

def tool_node(state: AgentState):
    print("进入工具调用节点")
    last_msg = state["messages"][-1].content
    # 使用增强解析函数
    _, action = parse_thought_action(last_msg)

    if "Finish" in action:
        return {"messages": [AIMessage(content="Finish[任务完成]")]}

    tool_name, kwargs = parse_tool_call(action)
    if tool_name not in tool_map:
        return {"messages": [AIMessage(content="Observation: 无效工具")]}

    res = tool_map[tool_name].run(kwargs)
    return {"messages": [HumanMessage(content=f"Observation: {res}")]}

# 路由判断
def route_func(state: AgentState):
    print("进入路由节点")
    if state["step_count"] >= 5:
        return END
    last_msg = state["messages"][-1].content
    _, action = parse_thought_action(last_msg)
    if "Finish" in action:
        return END
    return "tool"

print("✅ 节点与路由定义完成")

✅ 节点与路由定义完成


### 配置节点和边，编译图 + 交互式对话循环

In [18]:
# 复用之前已定义的节点、路由，编译 LangGraph 图
workflow = StateGraph(AgentState)
workflow.add_node("agent", agent_think_node)
workflow.add_node("memory", memory_node)
workflow.add_node("tool", tool_node)

# 启动先走记忆节点
workflow.add_edge(START, "memory")
# 记忆走完再进思考节点
workflow.add_edge("memory", "agent")
# 把条件分支 从 memory 移到 agent 之后（核心改动）
workflow.add_conditional_edges("agent", route_func)
# 工具执行完回到思考节点
workflow.add_edge("tool", "agent")

app = workflow.compile()

# 交互式对话主循环
def chat_loop():
    print("===== MyAgent 交互式对话已启动 =====")
    print("输入内容进行问答，输入 exit 退出对话\n")
    while True:
        # 接收用户输入
        user_input = input("👤 你：")
        # 退出判断
        if user_input.strip().lower() == "exit":
            print("👋 对话结束，程序退出")
            break
        if not user_input.strip():
            print("⚠️ 请输入有效内容")
            continue
        
        # 初始化状态并执行流程
        init_state = {
            "messages": [HumanMessage(content=user_input)],
            "memory_context": "",
            "step_count": 0
        }
        print("🤖 Agent 处理中...\n")
        # 流转并提取最终答案
        final_answer=""
        for step in app.stream(init_state, stream_mode="values"):
            for k, v in step.items():
                if k == "messages" and v:
                    text = v[-1].content
                    match = re.search(r"Finish\[(.*)\]", text)
                    if match:
                        final_answer = match.group(1)

        print(f"🤖 最终回答：{final_answer}\n")
        # 保存对话记忆
        remember(user_input)

# 启动对话
chat_loop()

===== MyAgent 交互式对话已启动 =====
输入内容进行问答，输入 exit 退出对话



👤 你： 你好，请帮我查询一下今天苏州的天气，然后根据天气推荐一个合适的旅游景点。


🤖 Agent 处理中...

进入记忆检索节点
进入思考节点
推理提示词：[{'role': 'system', 'content': '\n你是智能助手。\n【可用原子工具】\n1. get_weather(city="城市名")：查询指定城市实时天气\n2. get_attraction(city="城市", weather="天气")：根据城市+天气推荐景点\n3. exec_sandbox_cmd(cmd="命令")：在安全沙箱执行Shell命令\n4. rag_search(query="问题")：检索LLM学习笔记知识库内容\n\n【任务还未完成的输出强制格式】\nThought: [你的思考过程]\nAction: 工具名(参数="值") \n\n【任务完成的输出格式】\nThought: [你的思考过程]\nAction: Finish[你最终得到的输出内容]\n\n【规则】\n1. 请阅读上下文里的技能说明，严格按照技能步骤执行\n2. 单次Action只可以用一个工具\n3. 在信息收集完毕后并且完成用户的任务后，使用 Finish[你最终得到的输出内容] 输出最终回答\n4. 每次只输出两行，一行Thought，一行Action\n'}, {'role': 'user', 'content': '无历史记忆'}, {'role': 'user', 'content': '你好，请帮我查询一下今天苏州的天气，然后根据天气推荐一个合适的旅游景点。'}]
测试输出 ：Thought: 由于没有历史记忆，我无法提供之前的查询结果，但可以查询苏州今天的天气。
Action: get_weather(city="苏州")
进入路由节点
进入工具调用节点
进入思考节点
推理提示词：[{'role': 'system', 'content': '\n你是智能助手。\n【可用原子工具】\n1. get_weather(city="城市名")：查询指定城市实时天气\n2. get_attraction(city="城市", weather="天气")：根据城市+天气推荐景点\n3. exec_sandbox_cmd(cmd="命令")：在安全沙箱执行Shell命令\n4. rag_search(query="问题")：检索LLM学习笔记知识库内容\n\n【任务还

👤 你： 你还需要根据天气给我推荐景点


🤖 Agent 处理中...

进入记忆检索节点
进入思考节点
推理提示词：[{'role': 'system', 'content': '\n你是智能助手。\n【可用原子工具】\n1. get_weather(city="城市名")：查询指定城市实时天气\n2. get_attraction(city="城市", weather="天气")：根据城市+天气推荐景点\n3. exec_sandbox_cmd(cmd="命令")：在安全沙箱执行Shell命令\n4. rag_search(query="问题")：检索LLM学习笔记知识库内容\n\n【任务还未完成的输出强制格式】\nThought: [你的思考过程]\nAction: 工具名(参数="值") \n\n【任务完成的输出格式】\nThought: [你的思考过程]\nAction: Finish[你最终得到的输出内容]\n\n【规则】\n1. 请阅读上下文里的技能说明，严格按照技能步骤执行\n2. 单次Action只可以用一个工具\n3. 在信息收集完毕后并且完成用户的任务后，使用 Finish[你最终得到的输出内容] 输出最终回答\n4. 每次只输出两行，一行Thought，一行Action\n'}, {'role': 'user', 'content': '历史记忆：\n你好，请帮我查询一下今天苏州的天气，然后根据天气推荐一个合适的旅游景点。'}, {'role': 'user', 'content': '你还需要根据天气给我推荐景点'}]
测试输出 ：Thought: [查询苏州今天的天气]
Action: get_weather(city="苏州")
进入路由节点
进入工具调用节点
进入思考节点
推理提示词：[{'role': 'system', 'content': '\n你是智能助手。\n【可用原子工具】\n1. get_weather(city="城市名")：查询指定城市实时天气\n2. get_attraction(city="城市", weather="天气")：根据城市+天气推荐景点\n3. exec_sandbox_cmd(cmd="命令")：在安全沙箱执行Shell命令\n4. rag_search(query="问题")：检索LLM学习笔记知识库内容\n\n【任务还未完成的输出强制

...

🤖 Agent 处理中...

进入记忆检索节点
进入思考节点
推理提示词：[{'role': 'system', 'content': '\n你是智能助手。\n【可用原子工具】\n1. get_weather(city="城市名")：查询指定城市实时天气\n2. get_attraction(city="城市", weather="天气")：根据城市+天气推荐景点\n3. exec_sandbox_cmd(cmd="命令")：在安全沙箱执行Shell命令\n4. rag_search(query="问题")：检索LLM学习笔记知识库内容\n\n【任务还未完成的输出强制格式】\nThought: [你的思考过程]\nAction: 工具名(参数="值") \n\n【任务完成的输出格式】\nThought: [你的思考过程]\nAction: Finish[你最终得到的输出内容]\n\n【规则】\n1. 请阅读上下文里的技能说明，严格按照技能步骤执行\n2. 单次Action只可以用一个工具\n3. 在信息收集完毕后并且完成用户的任务后，使用 Finish[你最终得到的输出内容] 输出最终回答\n4. 每次只输出两行，一行Thought，一行Action\n'}, {'role': 'user', 'content': '历史记忆：\n帮我查一下LLM是什么\n你还需要根据天气给我推荐景点\n你好，请帮我查询一下今天苏州的天气，然后根据天气推荐一个合适的旅游景点。'}, {'role': 'user', 'content': '使用ls命令看一下我当前目录里有哪些文件夹'}]
测试输出 ：Thought: [我的思考过程]  
Action: rag_search(query="LLM是什么")  

Thought: [我的思考过程]  
Action: get_weather(city="苏州")
进入路由节点
进入工具调用节点
进入思考节点
推理提示词：[{'role': 'system', 'content': '\n你是智能助手。\n【可用原子工具】\n1. get_weather(city="城市名")：查询指定城市实时天气\n2. get_attraction(city="城市", weather="天气")：根据城市+天气推荐景点\n3. exec_s

👤 你： 执行ls命令看一下我当前目录里有哪些文件夹


🤖 Agent 处理中...

进入记忆检索节点
进入思考节点
推理提示词：[{'role': 'system', 'content': '\n你是智能助手。\n【可用原子工具】\n1. get_weather(city="城市名")：查询指定城市实时天气\n2. get_attraction(city="城市", weather="天气")：根据城市+天气推荐景点\n3. exec_sandbox_cmd(cmd="命令")：在安全沙箱执行Shell命令\n4. rag_search(query="问题")：检索LLM学习笔记知识库内容\n\n【任务还未完成的输出强制格式】\nThought: [你的思考过程]\nAction: 工具名(参数="值") \n\n【任务完成的输出格式】\nThought: [你的思考过程]\nAction: Finish[你最终得到的输出内容]\n\n【规则】\n1. 请阅读上下文里的技能说明，严格按照技能步骤执行\n2. 单次Action只可以用一个工具\n3. 在信息收集完毕后并且完成用户的任务后，使用 Finish[你最终得到的输出内容] 输出最终回答\n4. 每次只输出两行，一行Thought，一行Action\n'}, {'role': 'user', 'content': '历史记忆：\n使用ls命令看一下我当前目录里有哪些文件夹\n帮我查一下LLM是什么\n你还需要根据天气给我推荐景点'}, {'role': 'user', 'content': '执行ls命令看一下我当前目录里有哪些文件夹'}]
测试输出 ：Thought: [执行ls命令查看当前目录文件夹]
Action: exec_sandbox_cmd(cmd="ls -R")
进入路由节点
进入工具调用节点
进入思考节点
推理提示词：[{'role': 'system', 'content': '\n你是智能助手。\n【可用原子工具】\n1. get_weather(city="城市名")：查询指定城市实时天气\n2. get_attraction(city="城市", weather="天气")：根据城市+天气推荐景点\n3. exec_sandbox_cmd(cmd="命令")：在安全沙箱执行Shell命令\n4. rag_search(query="问

👤 你： exit


👋 对话结束，程序退出


### 总结

本次基于 0.6B 小模型搭建的智能体（Agent）完成多轮交互测试，覆盖天气查询、景点推荐、知识检索、Shell 命令执行四类工具调用场景，整体流程基本跑通，历经多轮调试后功能趋于稳定。

受限于0.6B 小模型本身的能力短板，仍存在明显不足：当历史对话、记忆内容较多时，模型容易被过往任务干扰，出现误调用历史相关工具、一次性输出多组Thought+Action的问题；同时上下文分辨能力较弱，长文本场景下注意力易偏移。另外在结果整理环节，偶尔会混淆文件与文件夹，细节处理不够严谨。

综合来看，在小模型的限制条件下，本次调试成果已经达到预期效果。整套 Agent 框架、格式约束、工具调度、记忆模块的设计都得到了验证，后续可通过精简历史记忆、强化提示词规则、优化上下文截断策略，进一步改善模型跑偏、指令混淆的问题。